# Raktári működés elemzése – Python backend

A notebook a teljes adatfeldolgozást bemutatja a betöltéstől a KPI-ig. Az eredménytáblákat a Power BI-riport tölti be.

**Bemenet:** a `02_Staging` mappa négy CSV-fájlja. Ezek a forrás-Excel 8 munkalapjából készültek, tartalmi változtatás nélkül (D-01, `D01_excel_to_csv.py`).

**Felépítés:**
1. Környezet, betöltés és kontroll
2. Tisztítás
3. Adatmodell (csillagséma)
4. ABC–XYZ elemzés (szedés)
5. Teljesítménypont
6. KPI
7. Export a Power BI-nak

A szövegekben a D-xx (döntés) és az A-xx (adatminőségi megállapítás) jelölések a munka során vezetett döntésnaplóm azonosítói (a repóban: `04_Dokumentacio/dontesnaplo.md`). Minden döntést és megállapítást ott fejtek ki, ahol a notebook felhasználja.

## 1. Környezet, betöltés és kontroll

In [1]:
# Környezet: könyvtárak és útvonalak
import platform
from pathlib import Path

import pandas as pd
import pyarrow  # a Parquet-exporthoz kell (D-26); ha telepítve van, a pandas a szövegoszlopokat is ezzel tárolja

MAPPA = Path.cwd().parent          # a projektmappa (a repó gyökere)
STAGING = MAPPA / "02_Staging"     # a négy CSV helye

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)
print("Staging mappa:", STAGING.relative_to(MAPPA), "| létezik:", STAGING.exists())

Python: 3.14.0
pandas: 3.0.6
pyarrow: 25.0.1
Staging mappa: 02_Staging | létezik: True


### 1.1 Betöltés

A négy CSV minden oszlopa szövegként töltődik be. Így a vezető nullás kódok (például a „020” gyártókód, A-13) nem alakulnak számmá. A típusokat a 2. szakasz állítja be, oszloponként kifejezetten megadva.

A CSV-ket a `D01_excel_to_csv.py` szkript készíti a forrás-Excelből (D-25). Futtatási sorrend: a forrás-Excel a `01_Forras` mappába kerül, utána a szkript, végül ez a notebook.

In [2]:
# 1.1 Betöltés: a négy CSV, minden oszlop szövegként
letszam   = pd.read_csv(STAGING / "letszam.csv",   sep=";", encoding="utf-8-sig", dtype=str)
betarolas = pd.read_csv(STAGING / "betarolas.csv", sep=";", encoding="utf-8-sig", dtype=str)
szedes    = pd.read_csv(STAGING / "szedes.csv",    sep=";", encoding="utf-8-sig", dtype=str)
tarhely   = pd.read_csv(STAGING / "tarhely.csv",   sep=";", encoding="utf-8-sig", dtype=str)

szedes.head()

,Cikkszam,Termekcsoportkodnev,Gyartokod,Termekcsoportkod,darab,ugyintezo,elkezdve,statusz,lezarva,teljesitmenypont,forras_lap
0,code4362112,Olajleeresztő csavar tömítés,753,6095,1,6368,2026-04-15 13:13:48,SZEDES,2026-04-15 13:36:51,2,Szedési teljesítmény 1
1,code1063041,Fékerő szabályzó,579,644,5,1,2026-04-17 06:39:01,SZEDES,2026-04-17 06:39:01,2,Szedési teljesítmény 1
2,code2167544,CRDi DELPHI nagyny szivattyú,L76,466,1,1,2026-04-17 06:35:25,SZEDES,2026-04-17 06:35:25,1,Szedési teljesítmény 1
3,code2561594,Kuplung hidraulika cső,P00,6027,1,1,2026-04-17 06:34:53,SZEDES,2026-04-17 06:34:53,1,Szedési teljesítmény 1
4,code4848662,Légszárító patron,P25,208,1,1,2026-04-17 06:37:32,SZEDES,2026-04-17 06:37:32,2,Szedési teljesítmény 1


### 1.2 Kontroll: a sorok és az oszlopok száma

A várt értékek a forrás-Excel munkalapjain lévő sorok száma (fejléc nélkül, A-01). Ugyanezeket a D01 szkript és a Power BI-betöltés is ellenőrizte. Ha a betöltött táblák ezekkel egyeznek, az Excel → CSV → notebook úton nem veszett el és nem duplázódott meg sor.

In [3]:
# 1.2 Kontroll: a sorok és az oszlopok száma a kontrollszámokkal (A-01)
KONTROLL_SOR = {"letszam": 27, "betarolas": 825_337, "szedes": 2_188_553, "tarhely": 834_925}
tablak = {"letszam": letszam, "betarolas": betarolas, "szedes": szedes, "tarhely": tarhely}

for nev, df in tablak.items():  # .items(), mert szótárról van szó, nem listáról, nev: a kulcs, df: az érték
    sor = len(df)
    allapot = "OK" if sor == KONTROLL_SOR[nev] else "ELTÉRÉS!"
    print(f"{nev:<10} {sor:>10,} sor (várt: {KONTROLL_SOR[nev]:>10,})  {len(df.columns):>2} oszlop  {allapot}")

letszam            27 sor (várt:         27)   4 oszlop  OK
betarolas     825,337 sor (várt:    825,337)  11 oszlop  OK
szedes      2,188,553 sor (várt:  2,188,553)  11 oszlop  OK
tarhely       834,925 sor (várt:    834,925)  14 oszlop  OK


### 1.3 Teljesen azonos sorok (A-05)

Két sor akkor számít azonosnak, ha a `forras_lap` kivételével minden oszlopuk egyezik. A `forras_lap` csak azt mutatja, melyik munkalapról jött a sor. Ha az összehasonlításban szerepelne, a különböző lapokon lévő példányok nem számítanának azonosnak (D-04). Ez a cella csak megszámolja az azonos sorokat, a törlés a 2. szakaszban történik. A várt érték (56 109) az adatfeltárás során mért szám (A-05). Power Queryben (`Table.Distinct`) is ellenőriztem: a duplikátumok eltávolítása után a betárolásban 816 642, a szedésben 2 141 139 sor marad, a tárhelyből egy sor sem esik ki.

In [4]:
# 1.3 Teljesen azonos sorok száma (A-05); összehasonlítás a forras_lap oszlop nélkül
KONTROLL_FELESLEGES = 56_109
osszesen = 0

for nev, df in tablak.items():
    oszlopok = df.columns.drop("forras_lap")
    felesleges = df.duplicated(subset=oszlopok).sum()
    felesleges_lappal = df.duplicated().sum()
    osszesen = osszesen + felesleges
    print(f"{nev:<10} {felesleges:>7,} felesleges sor  (a forras_lap oszloppal: {felesleges_lappal:>7,})")

allapot = "OK" if osszesen == KONTROLL_FELESLEGES else "ELTÉRÉS!"
print(f"Összesen: {osszesen:,} (várt: {KONTROLL_FELESLEGES:,})  {allapot}")

letszam          0 felesleges sor  (a forras_lap oszloppal:       0)
betarolas    8,695 felesleges sor  (a forras_lap oszloppal:   8,175)
szedes      47,414 felesleges sor  (a forras_lap oszloppal:  22,758)
tarhely          0 felesleges sor  (a forras_lap oszloppal:       0)
Összesen: 56,109 (várt: 56,109)  OK


## 2. Tisztítás

A tisztítás a döntésnapló döntéseit hajtja végre, ebben a sorrendben:

1. **Teljesen azonos sorok törlése** a betárolásban és a szedésben (D-04). Még a szöveges oszlopokon fut, ugyanúgy, ahogy az 1.3-as cella és a Power Query-s ellenőrzés számolta.
2. **A 9 lezáratlan szedési sor kizárása** (D-05). A típusváltás előtt kell, mert a „0000-00-00 00:00:00” nem érvényes dátum.
3. **Típusok** a típusterv szerint: egész számok és dátum/idő; a kódok szövegként maradnak.
4. **A 0 darabos sorok jelölése** (D-06).
5. **Az „1”-es ügyintézőkód ellenőrzése** (D-07).
6. **Tárhely:** készletkategória (Szabad / Tiltott, D-23) és a hiányzó tárhelytípus pótlása (A-15).
7. **Összesítő kontroll.**

Minden lépés után ellenőrző számítás fut: a sorok száma vagy egy összeg, a várt értékkel összevetve. A várt értékek a munkanaplóm kontrollszámai: ezeket az adatfeltárás során közvetlenül a forrás-Excelből számoltam.

### 2.1 Teljesen azonos sorok törlése (D-04)

Minden azonos sorból egy példány marad. Indoklás: a sorok minden mezőben egyeznek, a kezdés és a lezárás másodpercre azonos, és nincs olyan azonosító (például rendelésszám), amely külön eseményként megkülönböztetné őket. A duplikáció oka az adatból nem állapítható meg (A-05). A törlés a betárolási sorok kb. 1%-át, a szedési sorok kb. 2%-át érinti. A Létszámban és a Tárhelyben nincs azonos sor (1.3).

In [5]:
# 2.1: teljesen azonos sorok törlése; minden azonos sorból egy példány marad (D-04)
KONTROLL_D04 = {"betarolas": 816_642, "szedes": 2_141_139}

betarolas = betarolas.drop_duplicates(subset=betarolas.columns.drop("forras_lap"))
szedes    = szedes.drop_duplicates(subset=szedes.columns.drop("forras_lap"))

for nev, df in {"betarolas": betarolas, "szedes": szedes}.items():
    allapot = "OK" if len(df) == KONTROLL_D04[nev] else "ELTÉRÉS!"
    print(f"{nev:<10} {len(df):>10,} sor (várt: {KONTROLL_D04[nev]:>10,})  {allapot}")

betarolas     816,642 sor (várt:    816,642)  OK
szedes      2,141,139 sor (várt:  2,141,139)  OK


### 2.2 Lezáratlan szedési sorok kizárása (D-05)

9 szedési sorban a `lezarva` értéke „0000-00-00 00:00:00” (A-06). A lezáratlan tétel nem teljesítés, ezért ezek a sorok kimaradnak. Mind a 9998-as ügyintézőkódhoz tartoznak, amely nincs a Létszámban, és más sora sincs, így a kizárás a KPI-t és a teljesítménypontot nem érinti.

In [6]:
# 2.2 Lezáratlan szedési sorok kizárása (D-05)
KONTROLL_D05 = 2_141_130
lezaratlan = szedes["lezarva"] == "0000-00-00 00:00:00"  # maszk: True, ha a sor lezáratlan

print("Lezáratlan sorok:", lezaratlan.sum())
print("Ügyintézőkódjuk:", szedes.loc[lezaratlan, "ugyintezo"].unique().tolist()) # sorok és oszlopok kiválasztása: loc[sorok, oszlopok]
print("A 9998-as kód összes sora:", (szedes["ugyintezo"] == "9998").sum())

szedes = szedes[~lezaratlan]  # ~: Logikai tagadás, a maszkban False értékek maradnak meg

allapot = "OK" if len(szedes) == KONTROLL_D05 else "ELTÉRÉS!"
print(f"szedes {len(szedes):>10,} sor (várt: {KONTROLL_D05:>10,})  {allapot}")

Lezáratlan sorok: 9
Ügyintézőkódjuk: ['9998']
A 9998-as kód összes sora: 9
szedes  2,141,130 sor (várt:  2,141,130)  OK


### 2.3 Típusok

Az 1.1-es betöltés minden oszlopot szövegként olvasott be. Itt kapják meg a típusukat, oszloponként kifejezetten megadva:

- **Egész szám:** `darab` és `teljesitmenypont` (mozgás), `Darab` és a négy mm-oszlop (tárhely). A pandas `Int64` típusa a hiányzó értéket is kezeli: a tárhelysorok egy részénél nincs méret (A-15), ezek hiányzó értékként maradnak, nem 0-ként.
- **Dátum/idő:** `elkezdve` és `lezarva`, rögzített formátummal (ÉÉÉÉ-HH-NN óó:pp:mm, A-20).
- **Szöveg marad:** minden kód (`Cikkszam`, `Gyartokod`, `Termekcsoportkod`, `ugyintezo`, `ugyintezo_kod`, `Raktár`, `Tárhely`). Számként elveszne a vezető nulla (A-13), számolni pedig nem kell velük.

Az átalakítás szigorú: ha egy érték nem alakítható át, a cella hibával megáll, és nem hagy csendben üres értéket.

In [7]:
# 2.3 Típusok: egész szám (Int64) és dátum/idő; a kódok szövegként maradnak
EGESZ_MOZGAS   = ["darab", "teljesitmenypont"]
DATUM_MOZGAS   = ["elkezdve", "lezarva"]
EGESZ_TARHELY  = ["Tól (mm)", "Hossz (mm)", "Mélység (mm)", "Magasság (mm)", "Darab"]
DATUM_FORMATUM = "%Y-%m-%d %H:%M:%S"

for oszlop in EGESZ_MOZGAS:
    betarolas[oszlop] = pd.to_numeric(betarolas[oszlop]).astype("Int64")  # Int64 tud hiányzó értéket is tárolni, a sima int64 nem
    szedes[oszlop]    = pd.to_numeric(szedes[oszlop]).astype("Int64")  # igazából egy meglévő oszlop lecserélése a jobb oldalon létrehozottra

for oszlop in DATUM_MOZGAS:
    betarolas[oszlop] = pd.to_datetime(betarolas[oszlop], format=DATUM_FORMATUM)  # ha egy érték nem illik a mintára, hibával megáll (ezért volt előtte a 2.2)
    szedes[oszlop]    = pd.to_datetime(szedes[oszlop], format=DATUM_FORMATUM)

for oszlop in EGESZ_TARHELY:
    tarhely[oszlop] = pd.to_numeric(tarhely[oszlop]).astype("Int64")

print(szedes.dtypes, "\n")
print(tarhely.dtypes)

Cikkszam                          str
Termekcsoportkodnev               str
Gyartokod                         str
Termekcsoportkod                  str
darab                           Int64
ugyintezo                         str
elkezdve               datetime64[us]
statusz                           str
lezarva                datetime64[us]
teljesitmenypont                Int64
forras_lap                        str
dtype: object 

Raktár                   str
Tárhely típus            str
Tárhely                  str
Cikkszam                 str
Termekcsoportkodnev      str
Gyartokod                str
Termekcsoportkod         str
Tól (mm)               Int64
Hossz (mm)             Int64
Mélység (mm)           Int64
Magasság (mm)          Int64
Darab                  Int64
Tiltott?                 str
forras_lap               str
dtype: object


### 2.4 A 0 darabos sorok jelölése (D-06)

45 408 szedési sorban 0 a darab, a sor mégis kapott teljesítménypontot (A-07). Ezek a sorok maradnak, mert a darabalapú mutatókat nem torzítják. A tételsor-alapú ABC-elemzésből viszont kimaradnak, mert nem volt valódi kiszedés (4. szakasz). A jelölőoszlop (`nulla_darab`) mindkét mozgástáblába bekerül, hogy az összefűzött ténytáblában (3. szakasz) minden sornak legyen értéke.

In [8]:
# 2.4 A 0 darabos sorok jelölése (D-06); a sorok maradnak
KONTROLL_D06 = 45_408
betarolas["nulla_darab"] = betarolas["darab"] == 0  # új oszlop maszkolással: True, ha a darab értéke 0
szedes["nulla_darab"]    = szedes["darab"] == 0

allapot = "OK" if szedes["nulla_darab"].sum() == KONTROLL_D06 else "ELTÉRÉS!"
print("0 darabos betárolási sorok:", betarolas["nulla_darab"].sum())
print(f"0 darabos szedési sorok:    {szedes['nulla_darab'].sum():,} (várt: {KONTROLL_D06:,})  {allapot}")

0 darabos betárolási sorok: 0
0 darabos szedési sorok:    45,408 (várt: 45,408)  OK


### 2.5 Az „1”-es ügyintézőkód (D-07)

Az „1”-es kód nem szerepel a Létszámban. A cella azt ellenőrzi, hogy a soraiban a kezdés és a lezárás időpontja azonos-e. Ha igen, a sorok gépi könyvelésből származnak, mert ember nem végez szedést 0 másodperc alatt. A sorok valós keresletet mutatnak, ezért a mozgásadatokban és az ABC–XYZ-ben maradnak. A dolgozói kimutatásokból viszont kimaradnak: a DimDolgozo-ban (3. szakasz) „Rendszer” csoportba kerülnek.

In [9]:
# 2.5 Az „1”-es ügyintézőkód ellenőrzése (D-07)
egyes_kod = szedes["ugyintezo"] == "1"
idotartam = szedes["lezarva"] - szedes["elkezdve"]  # soronkénti időtartam

print(f"Az „1”-es kód szedési sorai:     {egyes_kod.sum():,}")
print(f"Ebből 0 másodperces:             {(idotartam[egyes_kod] == pd.Timedelta(0)).sum():,}") # idotartamok ahol egyes_kod=True van megegyeznek-e a 0-val, majd sum() amik megyegyeznek
print(f"Az „1”-es kód betárolási sorai:  {(betarolas['ugyintezo'] == '1').sum():,}")
print("Szerepel a Létszámban:", (letszam["ugyintezo_kod"] == "1").any())

Az „1”-es kód szedési sorai:     223,306
Ebből 0 másodperces:             223,306
Az „1”-es kód betárolási sorai:  0
Szerepel a Létszámban: False


### 2.6 Tárhely: készletkategória (D-23) és hiányzó tárhelytípus (A-15)

A `Tiltott?` oszlopban „TILTOTT” vagy üres érték áll. A tiltott sorok a sorok 48%-át és a készletdarab 39%-át adják. Ezért nem vágom ki őket, hanem külön kategóriát kapnak (Szabad / Tiltott), és a vizuálokon szűrővel kivehetők. A KPI-t nem érinti, mert a készletrész mindkét csoportnál a teljes 1/3 (D-14).

A 63–65., 68–75., 77–80. és 82-es raktárban nincs tárhelytípus és méret (125 654 sor, A-15). A hiányzó típus helyére „Nincs megadva” kerül, hogy a vizuálokon külön kategóriaként jelenjen meg. A méretek hiányzó értékként maradnak, mert a 0 valótlan méret lenne.

In [10]:
# 2.6 Tárhely: készletkategória (D-23) és hiányzó tárhelytípus (A-15)
tarhely["keszlet_kategoria"] = "Szabad"  # új oszlop
tarhely.loc[tarhely["Tiltott?"] == "TILTOTT", "keszlet_kategoria"] = "Tiltott" # a .loc bal oldalon ír, jobb oldalon olvas, itt a kiválasztott cellá(k)ba beíja a jobb oldali értéket, ha a bal oldali feltétel teljesül. Ahol a "Tiltott?" oszlop értéke "TILTOTT", ott a "keszlet_kategoria" oszlop értéke "Tiltott" lesz, egyébként "Szabad"
tarhely["Tárhely típus"] = tarhely["Tárhely típus"].fillna("Nincs megadva")  # a hiányzó értékeket kitölti "Nincs megadva"-val

tiltott = tarhely["keszlet_kategoria"] == "Tiltott"
tiltott_darab = tarhely.loc[tiltott, "Darab"].sum()
osszes_darab = tarhely["Darab"].sum()

print(f"Tiltott sorok:  {tiltott.sum():>9,} / {len(tarhely):>9,}  ({tiltott.mean():.0%})")
print(f"Tiltott darab:  {tiltott_darab:>9,} / {osszes_darab:>9,}  ({tiltott_darab / osszes_darab:.0%})")
print(f"Nincs megadva tárhelytípus: {(tarhely['Tárhely típus'] == 'Nincs megadva').sum():,} sor")

Tiltott sorok:    401,729 /   834,925  (48%)
Tiltott darab:  2,123,994 / 5,401,239  (39%)
Nincs megadva tárhelytípus: 125,654 sor


### 2.7 Összesítő kontroll

A tisztított táblák sorainak száma és összegei a döntésnapló „Tisztítás után” kontrollszámaival. A két, több táblát érintő kontrollszám (egyedi ügyintézőkódok és egyedi cikkszámok) a 3. szakaszba kerül, az összefűzött ténytábla és a cikktörzs elkészülte után.

In [11]:
# 2.7 Összesítő kontroll: a tisztított táblák a kontrollszámokkal
KONTROLL_TISZTA = {
    "Betárolási tételsorok":         816_642,
    "Betárolt darab":              3_175_566,
    "Betárolási teljesítménypont": 1_225_119,
    "Szedési tételsorok":          2_141_130,
    "Szedett darab":               3_210_184,
    "Szedési teljesítménypont":    3_211_882,
    "Tárhelysorok":                  834_925,
    "Készletdarab":                5_401_239,
}
mert = {
    "Betárolási tételsorok":       len(betarolas),
    "Betárolt darab":              betarolas["darab"].sum(),
    "Betárolási teljesítménypont": betarolas["teljesitmenypont"].sum(),
    "Szedési tételsorok":          len(szedes),
    "Szedett darab":               szedes["darab"].sum(),
    "Szedési teljesítménypont":    szedes["teljesitmenypont"].sum(),
    "Tárhelysorok":                len(tarhely),
    "Készletdarab":                tarhely["Darab"].sum(),
}

for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_TISZTA[nev] else "ELTÉRÉS!"
    print(f"{nev:<28} {ertek:>10,} (várt: {KONTROLL_TISZTA[nev]:>10,})  {allapot}")

Betárolási tételsorok           816,642 (várt:    816,642)  OK
Betárolt darab                3,175,566 (várt:  3,175,566)  OK
Betárolási teljesítménypont   1,225,119 (várt:  1,225,119)  OK
Szedési tételsorok            2,141,130 (várt:  2,141,130)  OK
Szedett darab                 3,210,184 (várt:  3,210,184)  OK
Szedési teljesítménypont      3,211,882 (várt:  3,211,882)  OK
Tárhelysorok                    834,925 (várt:    834,925)  OK
Készletdarab                  5,401,239 (várt:  5,401,239)  OK


## 3. Adatmodell (csillagséma)

A feladat három „Cél”-t ad meg: minden készletmozgási esemény egyértelműen hozzárendelhető legyen az azt végrehajtó dolgozóhoz, az érintett cikkszám törzsadataihoz és a dátum adataihoz. Ezt csillagsémával oldom meg: középen a ténytáblák (események és mennyiségek), körülöttük a dimenziótáblák (leíró adatok). Minden dimenzió egy kulcsoszlopon keresztül kapcsolódik a ténytáblához.

| Tábla | Típus | Egy sor | Kapcsolat |
|---|---|---|---|
| FactMozgas | ténytábla | egy betárolási vagy szedési tételsor | `ugyintezo` → DimDolgozo, `Cikkszam` → DimCikk, `Datum` → DimDatum |
| FactKeszlet | ténytábla | egy tárhely × cikkszám pár (pillanatkép) | `Cikkszam` → DimCikk |
| DimDolgozo | dimenzió | egy ügyintézőkód | kulcs: `ugyintezo` |
| DimCikk | dimenzió | egy cikkszám | kulcs: `Cikkszam` |
| DimDatum | dimenzió | egy nap (2026.04.01.–06.30.) | kulcs: `Datum` |

Minden kapcsolat egy-a-többhöz (1:*): a dimenzió egy sorához a ténytábla több sora tartozik, és a szűrés a dimenziótól a ténytábla felé hat. Előnyei: minden leíró adat egyetlen helyen van; a közös DimCikk-en keresztül egy termékcsoport-szűrő a mozgásokra és a készletre egyszerre hat; a Power BI-ban pedig nincs kétértelmű szűrési út. A FactKeszlet pillanatkép, nincs dátuma és ügyintézője, ezért csak a DimCikk-hez kapcsolódik (D-14).

A szakasz végén a 3.6-os cella ellenőrzi a három „Cél”-t: a ténytáblák minden kulcsértéke megvan-e a dimenzióban, és egyediek-e a dimenziókulcsok.

### 3.1 FactMozgas: betárolás és szedés egy ténytáblában (D-02, D-03)

A betárolás és a szedés oszlopai azonosak, ezért a két tábla egymás alá fűzhető, a folyamatot a `statusz` oszlop jelzi (BETAROLAS / SZEDES). Így egy kapcsolatkészlet elég, a két folyamat mutatói pedig szűréssel külön is számolhatók (D-02). A naptárhoz a lezárás napja köti a sort, mert az a teljesítés pillanata (D-03). Ehhez új oszlop készül: a `Datum` a `lezarva` napja 00:00:00 időponttal, mert a DimDatum kulcsa a nap. Az `elkezdve` és a `lezarva` változatlanul, időponttal együtt marad. A cikk leíró oszlopai (`Termekcsoportkodnev`, `Gyartokod`, `Termekcsoportkod`) a DimCikk-be kerülnek (A-12), a `forras_lap` pedig csak a betöltés ellenőrzéséhez kellett, ezért ezek kimaradnak a ténytáblából.

In [12]:
# 3.1 FactMozgas: a betárolás és a szedés egymás alá fűzve (D-02); naptárkulcs: a lezárás napja (D-03)
del tablak, df  # a nyers (1.2) és a köztes (2.1) táblákra mutató nevek törlése: innentől csak a tisztított táblák használhatók

LEIRO_OSZLOPOK = ["Termekcsoportkodnev", "Gyartokod", "Termekcsoportkod"]  # a DimCikk-be kerülnek (A-12)
KONTROLL_MOZGAS = {"BETAROLAS": 816_642, "SZEDES": 2_141_130, "Összesen": 2_957_772}

FactMozgas = pd.concat([betarolas, szedes], ignore_index=True)        # egymás alá fűzés, új sorcímkékkel
FactMozgas["Datum"] = FactMozgas["lezarva"].dt.normalize()             # a lezárás napja, 00:00:00 időponttal
FactMozgas = FactMozgas.drop(columns=LEIRO_OSZLOPOK + ["forras_lap"])  # a cikk leíró oszlopai és a forras_lap kivétele

mert = {
    "BETAROLAS": (FactMozgas["statusz"] == "BETAROLAS").sum(),
    "SZEDES":    (FactMozgas["statusz"] == "SZEDES").sum(),
    "Összesen":  len(FactMozgas),
}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_MOZGAS[nev] else "ELTÉRÉS!"
    print(f"{nev:<10} {ertek:>10,} sor (várt: {KONTROLL_MOZGAS[nev]:>10,})  {allapot}")
print("Dátumok:", FactMozgas["Datum"].min(), "–", FactMozgas["Datum"].max())

FactMozgas.head(3)

BETAROLAS     816,642 sor (várt:    816,642)  OK
SZEDES      2,141,130 sor (várt:  2,141,130)  OK
Összesen    2,957,772 sor (várt:  2,957,772)  OK
Dátumok: 2026-04-01 00:00:00 – 2026-06-23 00:00:00


,Cikkszam,darab,ugyintezo,elkezdve,statusz,lezarva,teljesitmenypont,nulla_darab,Datum
0,code912814,1,7834,2026-06-08 18:15:29,BETAROLAS,2026-06-08 18:16:29,1,False,2026-06-08
1,code244300,20,5627,2026-04-17 06:02:21,BETAROLAS,2026-04-17 06:03:21,2,False,2026-04-17
2,code3758861,2,4968,2026-04-18 17:18:53,BETAROLAS,2026-04-18 17:19:53,2,False,2026-04-18


### 3.2 DimDolgozo: minden ügyintézőkód, amely mozgást könyvelt (Cél 1; D-07, D-08, D-09)

A dolgozó-dimenzió kulcsa az ügyintézőkód, mert a név nem egyedi: két-két azonos nevű munkatárs van, eltérő kóddal (D-09). A tábla a FactMozgas összes különböző kódjából indul, nem a Létszámból, mert a mozgássorok 81%-át olyan kód könyvelte, amely nincs a Létszámban. Ha a dimenzió csak a Létszám 27 kódját tartalmazná, ezek a sorok nem kapcsolódnának dolgozóhoz, és nem teljesülne az „Egyes Cél”. A Létszám adatai (név, felettes) bal oldali illesztéssel kerülnek a kódok mellé. A csoport a felettesből adódik („Csoport 1” / „Csoport 2”). Az „1”-es kód „Rendszer” (gépi könyvelés, D-07), a többi kód „Csoporton kívüli” (D-08): ezek a mozgásokban és az ABC–XYZ-ben szerepelnek, a KPI-ban és a teljesítménypontban nem. A vizuálokhoz megjelenítő oszlop készül „Név (kód)” formában, hogy az azonos nevek ne olvadjanak össze (D-09). A Létszám mind a 27 kódjának van mozgása, így a két csoport létszáma 12 és 15 fő.

In [13]:
# 3.2 DimDolgozo: a mozgások összes ügyintézőkódja + a Létszám adatai (Cél 1; D-07, D-08, D-09)
KONTROLL_KOD = 559  # egyedi ügyintézőkódok száma a FactMozgas-ban
KONTROLL_CSOPORT = {"Csoport 1": 12, "Csoport 2": 15, "Rendszer": 1, "Csoporton kívüli": 531}

kodok = pd.DataFrame({"ugyintezo": FactMozgas["ugyintezo"].unique()})  # egyoszlopos tábla a szótárból készítve(pd.DataFrame miatt), minden kód egyszer
DimDolgozo = kodok.merge(letszam, how="left", left_on="ugyintezo", right_on="ugyintezo_kod", validate="one_to_one") # a merge LEFT JOIN szerű, a bal oldali tábla a kodok, a "one_to_one" ellenőrzi, hogy mindkét kulcs egyedi-e
DimDolgozo = DimDolgozo.drop(columns=["ugyintezo_kod", "forras_lap"])  # a kód már megvan (ugyintezo), a forras_lap nem kell

DimDolgozo["Csoport"] = "Csoporton kívüli"  # új oszlop, minden értéke "Csoporton kívüli"
DimDolgozo.loc[DimDolgozo["felettes"] == "Csoportvezető 1", "Csoport"] = "Csoport 1" # ahol a felettes oszlop értéke "Csoportvezető 1", ott "Csoport 1" kerül a Csoport oszlopba
DimDolgozo.loc[DimDolgozo["felettes"] == "Csoportvezető 2", "Csoport"] = "Csoport 2"
DimDolgozo.loc[DimDolgozo["ugyintezo"] == "1", "Csoport"] = "Rendszer"
DimDolgozo["Nev_kod"] = DimDolgozo["nev"].fillna(DimDolgozo["Csoport"]) + " (" + DimDolgozo["ugyintezo"] + ")" # (új oszlop, ahol) a név oszlopban a hiányzó értékeket a Csoport oszlop értékével tölti ki, majd a kódot is hozzáfűzi

allapot = "OK" if len(DimDolgozo) == KONTROLL_KOD else "ELTÉRÉS!"
print(f"Egyedi ügyintézőkódok: {len(DimDolgozo)} (várt: {KONTROLL_KOD})  {allapot}")
for csoport, vart in KONTROLL_CSOPORT.items():
    db = (DimDolgozo["Csoport"] == csoport).sum() # a csoporthoz tartozó sorok száma
    allapot = "OK" if db == vart else "ELTÉRÉS!"
    print(f"  {csoport:<17} {db:>4} (várt: {vart:>4})  {allapot}")

DimDolgozo.head()

Egyedi ügyintézőkódok: 559 (várt: 559)  OK
  Csoport 1           12 (várt:   12)  OK
  Csoport 2           15 (várt:   15)  OK
  Rendszer             1 (várt:    1)  OK
  Csoporton kívüli   531 (várt:  531)  OK


,ugyintezo,nev,felettes,Csoport,Nev_kod
0,7834,Németh Dóra,Csoportvezető 1,Csoport 1,Németh Dóra (7834)
1,5627,Szabó Dóra,Csoportvezető 2,Csoport 2,Szabó Dóra (5627)
2,4968,NaN,NaN,Csoporton kívüli,Csoporton kívüli (4968)
3,6935,NaN,NaN,Csoporton kívüli,Csoporton kívüli (6935)
4,6583,NaN,NaN,Csoporton kívüli,Csoporton kívüli (6583)


### 3.3 DimCikk: cikktörzs mindhárom forrásból (Cél 2; A-12, A-13, A-14, A-19)

A cikkszám leíró adatai (termékcsoportkód, termékcsoportnév, gyártókód) mindhárom forrásban szerepelnek, de a cikkszámok köre forrásonként eltér: 41 886 szedett cikkszám jelenleg nincs tárhelyen, 179 330 tárhelyen lévő cikkszámot pedig nem szedtek a három hónap alatt (A-19). Ha a cikktörzs csak egy forrásból készülne, a többi tábla egyes sorainak nem lenne párja. Ezért mindhárom forrásból készül: a négy oszlop egymás alá kerül, majd az ismétlődő sorokból egy marad. Ha egy cikkszámhoz forrásonként eltérő termékcsoport vagy gyártókód tartozna, a cikkszám több sorban maradna meg; a cella ezért ellenőrzi, hogy a `Cikkszam` egyedi (A-12). A termékcsoportnál a kód az azonosító, mert egy név két kóddal is szerepel („Külső tükör burkolat nélkül”: 760 és 826, A-14). A gyártókód szöveg marad, így a vezető nullák is megmaradnak: az összes forrásban 806 gyártókód van, ebből 55 kezdődik nullával (A-13). Az ABC–XYZ-kategória a 4. szakaszban kerül ebbe a táblába.

In [14]:
# 3.3 DimCikk: a három forrás cikkadatai egymás alá fűzve, ismétlődések nélkül (Cél 2; A-12, A-13, A-14, A-19)
CIKK_OSZLOPOK = ["Cikkszam", "Termekcsoportkod", "Termekcsoportkodnev", "Gyartokod"]
KONTROLL_CIKK = 450_149  # egyedi cikkszámok száma a három forrásból

cikkek = pd.concat([betarolas[CIKK_OSZLOPOK], szedes[CIKK_OSZLOPOK], tarhely[CIKK_OSZLOPOK]], ignore_index=True) # ha a szögletes zárójelbe egy oszlopnév helyett oszlopnevek listája kerül, az eredmény nem egy oszlop, hanem egy tábla ezekkel az oszlopokkal
DimCikk = cikkek.drop_duplicates(ignore_index=True)  # a négy oszlopban azonos sorokból egy marad

allapot = "OK" if len(DimCikk) == KONTROLL_CIKK else "ELTÉRÉS!"
print(f"Egymás alá fűzve:     {len(cikkek):>9,} sor")
print(f"Ismétlődések nélkül:  {len(DimCikk):>9,} sor (várt: {KONTROLL_CIKK:,})  {allapot}")
print("A Cikkszam egyedi (A-12):", DimCikk["Cikkszam"].is_unique)  # True, ha a Cikkszam oszlopban nincs ismétlődő érték, ez kell a 1:*-hez
print(f"Termékcsoportkódok: {DimCikk['Termekcsoportkod'].nunique():,}, termékcsoportnevek: {DimCikk['Termekcsoportkodnev'].nunique():,} (A-14)")  # egy név két gyártókóddal szerepel
gyartokodok = DimCikk["Gyartokod"].drop_duplicates()  # a különböző gyártókódok, mindegyik egyszer
print(f"Gyártókódok: {len(gyartokodok)}, ebből vezető nullás: {gyartokodok.str.startswith('0').sum()} (A-13)")

DimCikk.head()

Egymás alá fűzve:     3,792,697 sor
Ismétlődések nélkül:    450,149 sor (várt: 450,149)  OK
A Cikkszam egyedi (A-12): True
Termékcsoportkódok: 1,157, termékcsoportnevek: 1,156 (A-14)
Gyártókódok: 806, ebből vezető nullás: 55 (A-13)


,Cikkszam,Termekcsoportkod,Termekcsoportkodnev,Gyartokod
0,code912814,377,Forgattyúsház fedél,H78
1,code244300,429,Gázos (Cofap) lengéscsillapító,P94
2,code3758861,286,Üzemanyagnyomás-szabályzó,I98
3,code4640689,556,Rozsdaoldó - jövedéki,020
4,code202494,1244,Tartófal,L64


### 3.4 DimDatum: naptártábla (Cél 3)

A naptártábla minden napot egyszer tartalmaz, a három hónap első napjától az utolsóig (2026.04.01.–06.30.), akkor is, ha az adott napon nem volt mozgás. Az adatok június 23-ig tartanak, de a teljes június szerepel, így a naptár hézagmentes; a Power BI időintelligencia-függvényei ezt igénylik. A napból számolt oszlopok (év, hónap, hét, a hét napja) a vizuálok szűrőihez és tengelyeihez kellenek. A hónap- és a napnevek magyarul, rögzített listából készülnek, hogy ne függjenek a gép területi beállításától. Munkanap-jelölő helyett hétvége-jelölő készül, mert a raktár hétvégén is dolgozik (a szedésnek áprilisban mind a 30 napján van adata).

In [15]:
# 3.4 DimDatum: minden nap 2026.04.01. és 06.30. között, egy sor = egy nap (Cél 3)
KONTROLL_NAP = 91  # 30 + 31 + 30
HONAPNEVEK = {4: "április", 5: "május", 6: "június"}  # a két szükséges szótár
NAPNEVEK = {1: "hétfő", 2: "kedd", 3: "szerda", 4: "csütörtök", 5: "péntek", 6: "szombat", 7: "vasárnap"}

DimDatum = pd.DataFrame({"Datum": pd.date_range("2026-04-01", "2026-06-30", freq="D")}) # a pd.date_range() dátumsorozatot készít a kezdőnaptól a zárónapig, mindkettőt beleértve, freq="D" = naponta; ebből pedig a pd.DataFrame() egy oszlopos táblát készít
DimDatum["Ev"]          = DimDatum["Datum"].dt.year  # a Datum oszlopból a dt.year attribútummal az évszámot kinyerjük, majd az új Ev oszlopba tesszük
DimDatum["Honap"]       = DimDatum["Datum"].dt.month
DimDatum["HonapNev"]    = DimDatum["Honap"].map(HONAPNEVEK)  # a map() a szótárban keresi a hónap számához tartozó nevet, majd az új HonapNev oszlopba teszi
DimDatum["Het"]         = DimDatum["Datum"].dt.isocalendar().week.astype("int32")  # ISO-hét, hétfőn kezdődik, típusváltás Power BI-hoz
DimDatum["Nap"]         = DimDatum["Datum"].dt.day
DimDatum["HetNapja"]    = DimDatum["Datum"].dt.dayofweek + 1  # 1 = hétfő ... 7 = vasárnap (+1, hogy ne 0-6 legyen)
DimDatum["HetNapjaNev"] = DimDatum["HetNapja"].map(NAPNEVEK)  # a napok számához tartozó név a szótárból
DimDatum["Hetvege"]     = DimDatum["HetNapja"] >= 6  # True: hétvége

allapot = "OK" if len(DimDatum) == KONTROLL_NAP else "ELTÉRÉS!"
print(f"Napok: {len(DimDatum)} (várt: {KONTROLL_NAP})  {allapot}")
print("Első és utolsó nap:", DimDatum["Datum"].min(), "–", DimDatum["Datum"].max())
print(DimDatum.dtypes)
DimDatum.head(7)

Napok: 91 (várt: 91)  OK
Első és utolsó nap: 2026-04-01 00:00:00 – 2026-06-30 00:00:00
Datum          datetime64[us]
Ev                      int32
Honap                   int32
HonapNev                  str
Het                     int32
Nap                     int32
HetNapja                int32
HetNapjaNev               str
Hetvege                  bool
dtype: object


,Datum,Ev,Honap,HonapNev,Het,Nap,HetNapja,HetNapjaNev,Hetvege
0,2026-04-01,2026,4,április,14,1,3,szerda,False
1,2026-04-02,2026,4,április,14,2,4,csütörtök,False
2,2026-04-03,2026,4,április,14,3,5,péntek,False
3,2026-04-04,2026,4,április,14,4,6,szombat,True
4,2026-04-05,2026,4,április,14,5,7,vasárnap,True
5,2026-04-06,2026,4,április,15,6,1,hétfő,False
6,2026-04-07,2026,4,április,15,7,2,kedd,False


### 3.5 FactKeszlet: a tárhelyek tartalma (D-14, D-23)

A tárhelytábla a legfrissebb állapot pillanatképe (D-14). Egy sora egy cikkszám egy tárhelyen, a darabszámával. Egy tárhelyen több cikkszám, egy cikkszám pedig több tárhelyen is lehet, de a tárhely–cikkszám pár mind a 834 925 sorban egyedi, így a `Darab` dupla számolás nélkül összeadható. Dátuma és ügyintézője nincs, ezért csak a DimCikk-hez kapcsolódik. A cikk leíró oszlopai itt is a DimCikk-be kerülnek, a `Tiltott?` helyett pedig a 2.6-os `keszlet_kategoria` marad (D-23).

In [16]:
# 3.5 FactKeszlet: egy sor = egy tárhely × cikkszám pár, a legfrissebb állapot pillanatképe (D-14, D-23)
KONTROLL_KESZLET = {"Sorok": 834_925, "Készletdarab": 5_401_239, "Ismétlődő tárhely × cikkszám pár": 0}

FactKeszlet = tarhely.drop(columns=LEIRO_OSZLOPOK + ["Tiltott?", "forras_lap"])  # a Tiltott? helyett a keszlet_kategoria marad (2.6)

mert = {
    "Sorok":                            len(FactKeszlet),  # az oszlopok elhagyásával sorok nem vesztek el
    "Készletdarab":                     FactKeszlet["Darab"].sum(),
    "Ismétlődő tárhely × cikkszám pár": FactKeszlet.duplicated(subset=["Tárhely", "Cikkszam"]).sum(),
}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_KESZLET[nev] else "ELTÉRÉS!"
    print(f"{nev:<33} {ertek:>10,} (várt: {KONTROLL_KESZLET[nev]:>10,})  {allapot}")

FactKeszlet.head(3)

Sorok                                834,925 (várt:    834,925)  OK
Készletdarab                       5,401,239 (várt:  5,401,239)  OK
Ismétlődő tárhely × cikkszám pár           0 (várt:          0)  OK


,Raktár,Tárhely típus,Tárhely,Cikkszam,Tól (mm),Hossz (mm),Mélység (mm),Magasság (mm),Darab,keszlet_kategoria
0,51,F,510101010100,code4084751,0,880,500,330,12,Szabad
1,51,F,510101010200,code3862458,0,780,500,330,15,Szabad
2,51,F,510101010300,code845675,0,770,500,330,15,Szabad


### 3.6 A három „Cél” ellenőrzése

Egy kapcsolat akkor egyértelmű, ha a ténytábla minden kulcsértéke megvan a dimenzióban (nincs olyan sor, amely a Power BI-ban üres dimenzióértékhez kerülne), és a dimenzió kulcsa egyedi (különben a kapcsolat nem lehet 1:*). A cella mindkettőt ellenőrzi mind a négy kapcsolatra. Utána megszámolja, hány különböző cikkszám szerepel a mozgásban, a készletben és a szedésben (A-12, A-19).

In [17]:
# 3.6 A három „Cél” ellenőrzése: minden ténytábla-kulcs megvan a dimenzióban, és a dimenziókulcsok egyediek
celok = {
    "Cél 1: FactMozgas.ugyintezo → DimDolgozo": FactMozgas["ugyintezo"].isin(DimDolgozo["ugyintezo"]).all(), # a bal oldali oszlop minden sorára megnézi, hogy az érték előfordul-e a zárójelben megadott oszlopban, bárhol. Az eredmény egy True/False oszlop (maszk), ugyanannyi sorral, mint a FactMozgas (2 957 772).
    "Cél 2: FactMozgas.Cikkszam → DimCikk":     FactMozgas["Cikkszam"].isin(DimCikk["Cikkszam"]).all(), # .all() True, ha a maszk minden eleme True, azaz minden cikkszám szerepel a DimCikk-ben
    "Cél 3: FactMozgas.Datum → DimDatum":       FactMozgas["Datum"].isin(DimDatum["Datum"]).all(),
    "FactKeszlet.Cikkszam → DimCikk":           FactKeszlet["Cikkszam"].isin(DimCikk["Cikkszam"]).all(),
    "DimDolgozo kulcsa egyedi":                 DimDolgozo["ugyintezo"].is_unique,
    "DimCikk kulcsa egyedi":                    DimCikk["Cikkszam"].is_unique,
    "DimDatum kulcsa egyedi":                   DimDatum["Datum"].is_unique,
}
for nev, rendben in celok.items():
    print(f"{nev:<42} {'OK' if rendben else 'HIBA!'}")

# Fordított irány: hány különböző cikkszám szerepel a mozgásban, a készletben és a szedésben (A-12, A-19)
KONTROLL_CIKK_BONTAS = {"Mozgásban": 295_990, "Készleten": 407_722, "Szedésben": 270_278}
mert = {
    "Mozgásban": FactMozgas["Cikkszam"].nunique(), # hány különböző cikkszám szerepel  a FactMozgas-ban
    "Készleten": FactKeszlet["Cikkszam"].nunique(),
    "Szedésben": FactMozgas.loc[FactMozgas["statusz"] == "SZEDES", "Cikkszam"].nunique(), # ahol a "statusz" oszlop értéke "SZEDES", ott unique-ok száma a "Cikkszam" oszlopban
}
print()
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_CIKK_BONTAS[nev] else "ELTÉRÉS!"
    print(f"{nev:<10} {ertek:>9,} különböző cikkszám (várt: {KONTROLL_CIKK_BONTAS[nev]:>9,})  {allapot}")

Cél 1: FactMozgas.ugyintezo → DimDolgozo   OK
Cél 2: FactMozgas.Cikkszam → DimCikk       OK
Cél 3: FactMozgas.Datum → DimDatum         OK
FactKeszlet.Cikkszam → DimCikk             OK
DimDolgozo kulcsa egyedi                   OK
DimCikk kulcsa egyedi                      OK
DimDatum kulcsa egyedi                     OK

Mozgásban    295,990 különböző cikkszám (várt:   295,990)  OK
Készleten    407,722 különböző cikkszám (várt:   407,722)  OK
Szedésben    270,278 különböző cikkszám (várt:   270,278)  OK


## 4. ABC–XYZ elemzés (szedés)

A feladat a termékek ABC- és XYZ-kategóriáját kéri a szedési adatok alapján; a határértékeket szabadon lehet megválasztani. A két besorolás ugyanazt az alapadatot vizsgálja két szempontból, cikkszámonként:

- **ABC – mennyi:** a három hónap összes szedése. A cikkszámok csökkenő sorrendben összeadódnak. Az A-osztály az a cikkkör, amely együtt a szedések első 80%-át adja, a B a következő 15%-ot, a C az utolsó 5%-ot (D-15).
- **XYZ – mennyire egyenletes:** a heti szedések ingadozása. A mérőszám a relatív szórás (CV = a heti értékek szórása / a heti értékek átlaga). X: CV ≤ 0,5; Y: 0,5 < CV ≤ 1,0; Z: CV > 1,0 (D-16).

**Az alap a szedési tételsor.** A szedési tábla egy sora egy elvégzett szedés: egy cikkszámból egy alkalommal kivett mennyiség. Mindkét besorolás a tételsorok számán alapul, nem a darabon, mert a szedés munkája soronként egyszer jelentkezik, a kivett mennyiségtől függetlenül; a feladatkiíró is a szedések számát erősítette meg (D-15). A 0 darabos sorok kimaradnak, mert nem volt valódi kiszedés (D-06). Az „1”-es kód (gépi könyvelés) sorai bent maradnak, mert valós keresletet mutatnak (D-07). A darab-kiugrók a tételsor-alapú számolást nem torzítják, ezért nem kell kizárni őket (D-19).

**Döntések:**
- **ABC-határok:** 80 / 15 / 5% kumulált tételsorarány. Az azonos tételsorszámú cikkszámok azonos osztályba kerülnek: az osztály alsó határa annak a cikkszámnak a tételsorszáma, amelyik átlépi a 80%-ot, illetve a 95%-ot. Különben két azonos forgalmú cikkszám csak a sorrend miatt kerülhetne különböző osztályba (D-15).
- **XYZ-időegység:** heti bontás a 11 teljes ISO-héten (15–25.). A 14. hét (04.01–04.05.) és a 26. hét (06.22–06.23.) csonka; az alacsony értékük ingadozásként jelenne meg, ezért kimaradnak. Az ABC a teljes időszakot használja (D-16).
- **Teljes rács:** a heti értékek egy cikkszám × hét táblából jönnek, amelyben a szedés nélküli hét értéke 0. A nullák nélkül egy ritkán szedett cikkszám egyenletesnek látszana (D-16).
- **Szórás:** a 11 hétre számolt populációs szórás, mert a 11 hét a teljes vizsgált időszak, nem minta. A csak a csonka heteken szedett cikkszámok CV-je nem számolható (0/0), ezek Z-be kerülnek (D-16).
- **Hely:** az osztályok statikus oszlopként a DimCikk-be kerülnek, mert a három hónap alapján a cikk tulajdonságai, és a Power BI szűrői nem változtathatják meg őket. A nem szedett cikkszámok „Nincs szedés” kategóriát kapnak (D-17).

**A szakasz lépései:** 4.1 a szedési alap; 4.2 tételsorok cikkszámonként; 4.3 ABC; 4.4 a heti rács; 4.5 XYZ; 4.6 az osztályok a DimCikk-be; 4.7 az ABC–XYZ mátrix; 4.8 a készlet ABC-osztályonként; 4.9 az eredmény értelmezése.

### 4.1 Az ABC–XYZ alapja: a valódi szedési tételsorok (D-06, D-07)

A FactMozgas szedési sorai közül azok maradnak, amelyekben a darab nem 0 (D-06); az „1”-es kód sorai bent maradnak (D-07). A táblába csak a további lépésekhez szükséges oszlopok kerülnek: a `Cikkszam` (erre csoportosít az ABC és az XYZ), az `ugyintezo` (az „1”-es kód sorainak megszámolásához) és a `Datum`. A hét a naptártáblából (DimDatum) jön, a `Datum` szerinti illesztéssel, így ugyanazt az ISO-hetet használja, mint a Power BI-modell. A kontrollszám a tisztított szedési sorok (2.7) és a 0 darabos szedési sorok (2.4) különbsége.

In [18]:
# 4.1 Az ABC–XYZ alapja: a szedési tételsorok, a 0 darabos sorok nélkül (D-06, D-07)
KONTROLL_ABC_SOR = 2_141_130 - 45_408  # a tisztított szedési sorok (2.7) mínusz a 0 darabos szedési sorok (2.4)

valodi_szedes = (FactMozgas["statusz"] == "SZEDES") & ~FactMozgas["nulla_darab"]  # a maszk sorai a szedési tételsorok, a 0 darabos sorok nélkül
szedes_abc = FactMozgas.loc[valodi_szedes, ["Cikkszam", "ugyintezo", "Datum"]]   # a maszk sorai, három oszloppal # a maszkban True értékek maradnak meg, a szögletes zárójelben a három oszlop neve van, így csak ezek az oszlopok maradnak meg
szedes_abc = szedes_abc.merge(DimDatum[["Datum", "Het"]], how="left", on="Datum", validate="many_to_one")  # a hét a naptártáblából (3.4)

cikk = szedes_abc["Cikkszam"].nunique() # hány különböző cikkszám szerepel a szedési tételsorokban, a 0 darabos sorok nélkül
allapot = "OK" if len(szedes_abc) == KONTROLL_ABC_SOR else "ELTÉRÉS!"
print(f"Szedési tételsorok a 0 darabosak nélkül: {len(szedes_abc):>9,} (várt: {KONTROLL_ABC_SOR:,})  {allapot}")
print(f"Hét nélküli sorok (dátum a naptáron kívül): {szedes_abc['Het'].isna().sum():>6,}") # a szedési tételsorok közül hány sorhoz nem talált a merge a DimDatum táblában dátumot, azaz a Het oszlopban NaN érték van
print(f"Az „1”-es kód (Rendszer) sorai:          {(szedes_abc['ugyintezo'] == '1').sum():>9,}")
print(f"Különböző cikkszám:                      {cikk:>9,} (3.6: {KONTROLL_CIKK_BONTAS['Szedésben']:,} szedett; csak 0 darabos szedése volt: {KONTROLL_CIKK_BONTAS['Szedésben'] - cikk:,})")
szedes_abc.head()

Szedési tételsorok a 0 darabosak nélkül: 2,095,722 (várt: 2,095,722)  OK
Hét nélküli sorok (dátum a naptáron kívül):      0
Az „1”-es kód (Rendszer) sorai:            208,163
Különböző cikkszám:                        269,737 (3.6: 270,278 szedett; csak 0 darabos szedése volt: 541)


,Cikkszam,ugyintezo,Datum,Het
0,code4362112,6368,2026-04-15,16
1,code1063041,1,2026-04-17,16
2,code2167544,1,2026-04-17,16
3,code2561594,1,2026-04-17,16
4,code4848662,1,2026-04-17,16


### 4.2 Tételsorok cikkszámonként (D-15)

Az ABC alapszáma cikkszámonként a szedések (tételsorok) száma a három hónap alatt. A `szedes_abc` sorai cikkszám szerint csoportosulnak, és minden csoportban megszámolódik a sorok száma. Az eredmény egy új tábla: egy sora egy szedett cikkszám, a `Tetelsor` oszlopban a szedéseinek számával. Ez a teljes időszakot használja, a csonka hetekkel együtt. Két kontroll ellenőrzi, hogy a csoportosításban egy sor sem veszett el, és egy sem számít kétszer: a tábla sorainak száma egyezik a 4.1-ben mért különböző cikkszámokkal, a `Tetelsor` összege pedig a 4.1-es tábla sorainak számával.

In [19]:
# 4.2 Tételsorok cikkszámonként: egy sor = egy szedett cikkszám (D-15)
abc = szedes_abc.groupby("Cikkszam").size().reset_index(name="Tetelsor")  # a .groupby() metódus a sorokat cikkszám szerint csoportokba osztja; a .size() csoportonként megszámolja a sorokat, az eredmény egy egyoszlopos sorozat (Series), a cikkszám a sorcímkébe kerül, nem oszlopba; a .reset_index() a sorozatot (Series-t) táblává alakítja, a name=... az új oszlop nevét adja meg, a sorcímkét (=a cikkszámot) visszateszi rendes oszlopba, a sorcímke pedig új, 0-tól induló sorszámozás lesz.

KONTROLL_ABC_ALAP = {"Cikkszámok": cikk, "Tételsorok": len(szedes_abc)}  # a 4.1 eredményei
mert = {"Cikkszámok": len(abc), "Tételsorok": abc["Tetelsor"].sum()} # a 4.2 eredményei
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_ABC_ALAP[nev] else "ELTÉRÉS!"
    print(f"{nev:<11} {ertek:>9,} (várt: {KONTROLL_ABC_ALAP[nev]:>9,})  {allapot}")

print(f"\nTételsor cikkszámonként: medián {abc['Tetelsor'].median():.0f}, átlag {abc['Tetelsor'].mean():.1f}, legtöbb {abc['Tetelsor'].max():,}")
print(f"Csak egyszer szedett cikkszám: {(abc['Tetelsor'] == 1).sum():,}")
abc.head()

Cikkszámok    269,737 (várt:   269,737)  OK
Tételsorok  2,095,722 (várt: 2,095,722)  OK

Tételsor cikkszámonként: medián 3, átlag 7.8, legtöbb 2,522
Csak egyszer szedett cikkszám: 84,594


,Cikkszam,Tetelsor
0,code1000003,6
1,code1000024,4
2,code100004,3
3,code1000058,1
4,code1000075,1


### 4.3 ABC: rangsor, kumulált arány, osztály (D-15)

A cikkszámok a tételsoraik száma szerint csökkenő sorrendbe kerülnek; azonos tételsorszámnál a cikkszám dönti el a sorrendet, hogy a rangsor minden futtatáskor ugyanaz legyen. A kumulált arány megmutatja, hogy az adott cikkszám és az összes előtte álló együtt a szedések hányad részét adja. A 80%-os és a 95%-os határt tételsorszámra fordítom: annak a cikkszámnak a tételsorszáma, amelyik átlépi a határt, lesz az osztály alsó határa. Így minden azonos tételsorszámú cikkszám ugyanabba az osztályba kerül, és a besorolás nem függ a cikkszámok ábécérendjétől (D-15).

In [20]:
# 4.3 ABC: rangsor, kumulált arány, osztály (D-15)
ABC_HATAR = {"A": 0.80, "B": 0.95}  # kumulált tételsorarány: A az első 80%, B a következő 15%, C a maradék 5%
KONTROLL_ABC = {"A": 76_209, "B": 108_934, "C": 84_594}

abc = abc.sort_values(["Tetelsor", "Cikkszam"], ascending=[False, True], ignore_index=True)  # csökkenő tételsor; azonosnál cikkszám szerint (ábécé szerint) [ez azért kell, hogy a rangsor minden futtatáskor azonos legyen]; az új sorszámok 0-tól indulnak, a 0 a legtöbbször szedett cikkszám
abc["Kum_arany"] = abc["Tetelsor"].cumsum() / abc["Tetelsor"].sum()  # futó összeg: minden sorba az addigi értékek összege kerül, az elsőtől az aktuálisig / összes tételsor (2 095 722)

min_A = abc.loc[abc["Kum_arany"] > ABC_HATAR["A"], "Tetelsor"].max()  # a 80%-ot átlépő cikkszám tételsorainak száma; ugye 0-tól indul a Kum_arany, így a 80%-ot átlépő sor az első, amelyik már 0.8001 vagy nagyobb, és annak a Tetelsor értéke kell. (Itt a .loc keresi a sorokat, ahol a feltétel teljesül, majd a "Tetelsor" oszlop értékét adja vissza, és a .max() a legnagyobb értéket adja vissza ezek közül.)
min_B = abc.loc[abc["Kum_arany"] > ABC_HATAR["B"], "Tetelsor"].max()  # a 95%-ot átlépő cikkszám tételsorainak száma; ugye 0-tól indul a Kum_arany, így a 95%-ot átlépő sor az első, amelyik már 0.9501 vagy nagyobb, és annak a Tetelsor értéke kell
abc["ABC"] = "C"
abc.loc[abc["Tetelsor"] >= min_B, "ABC"] = "B" # Itt a .loc már ír: a bal oldali feltétel teljesülése esetén a jobb oldali oszlopba (ABC) beírja a "B" értéket, azaz a Tetelsor értéke nagyobb vagy egyenlő, mint a min_B (a 95%-ot átlépő sor tételsora)
abc.loc[abc["Tetelsor"] >= min_A, "ABC"] = "A" # sorrendiség számít: előbb a B, majd az A, így az A osztályba kerülnek a legnagyobb tételsorok, a B-be a közepesek, a C-be a maradék. (Ha fordítva lenne, akkor a B-be kerülne minden, ami nagyobb vagy egyenlő, mint min_B, és az A osztály üres lenne.)

osszes = abc["Tetelsor"].sum()
print(f"Határok tételsorban: A ≥ {min_A}, B {min_B}–{min_A - 1}, C < {min_B}")
print(f"A 80%-os határon {(abc['Tetelsor'] == min_A).sum():,} cikkszámnak van pontosan {min_A} tételsora: mind A")
print(f"A 95%-os határon {(abc['Tetelsor'] == min_B).sum():,} cikkszámnak van pontosan {min_B} tételsora: mind B\n")
for osztaly, vart in KONTROLL_ABC.items():
    resz = abc.loc[abc["ABC"] == osztaly] # a .loc a feltételnek megfelelő sorokat adja vissza, az ABC oszlop értéke megegyezik az osztálynévvel
    allapot = "OK" if len(resz) == vart else "ELTÉRÉS!"
    print(f"{osztaly}: {len(resz):>7,} cikkszám ({len(resz) / len(abc):5.1%})  {resz['Tetelsor'].sum():>9,} tételsor ({resz['Tetelsor'].sum() / osszes:5.1%})  (várt: {vart:,})  {allapot}")
abc.head()

Határok tételsorban: A ≥ 6, B 2–5, C < 2
A 80%-os határon 10,890 cikkszámnak van pontosan 6 tételsora: mind A
A 95%-os határon 46,544 cikkszámnak van pontosan 2 tételsora: mind B

A:  76,209 cikkszám (28.3%)  1,683,312 tételsor (80.3%)  (várt: 76,209)  OK
B: 108,934 cikkszám (40.4%)    327,816 tételsor (15.6%)  (várt: 108,934)  OK
C:  84,594 cikkszám (31.4%)     84,594 tételsor ( 4.0%)  (várt: 84,594)  OK


,Cikkszam,Tetelsor,Kum_arany,ABC
0,code2110557,2522,0.001203,A
1,code3862458,1783,0.002054,A
2,code3427554,1498,0.002769,A
3,code5394324,1161,0.003323,A
4,code545631,1133,0.003864,A


### 4.4 XYZ: a heti rács, a nulla hetekkel együtt (D-16)

Az XYZ cikkszámonként a heti szedések számát vizsgálja. A rács egy tábla, amelynek sorai a cikkszámok, oszlopai a hetek, egy cellája pedig azt mutatja, hányszor szedték az adott cikkszámot az adott héten. Ahol egy héten nem volt szedés, a cella értéke 0: a forrásadatban erre a hétre nincs sor, a rács viszont minden cikkszám × hét párt tartalmaz. A rács előbb az összes hétre készül, és csak utána maradnak meg a teljes hetek (15–25.) oszlopai; így a csak a csonka heteken szedett cikkszámok is a rácsban maradnak, csupa 0 értékkel (D-16). A kontroll: a rácsnak annyi sora van, ahány szedett cikkszám, 11 oszlopa, és a cellák összege egyezik a teljes heteken szedett tételsorok számával.

In [21]:
# 4.4 XYZ: heti tételsorok a teljes cikkszám × hét rácson, a nulla hetekkel együtt (D-16)
TELJES_HETEK = list(range(15, 26))  # a 15–25. ISO-hét (a számokat adja 15-től 25-ig: a felső határ nincs benne, ez a Python szabálya); a 14. (04.01–04.05.) és a 26. (06.22–06.23.) csonka; lista készül [15, 16, ..., 25] elemekkel

racs = pd.crosstab(szedes_abc["Cikkszam"], szedes_abc["Het"])  # sor: cikkszám (az első érték), oszlop: hét (a második érték), cella: a tételsorok száma minden cikkszám × hét kombinációra (0, ha nem volt); a sorok ábécé sorrendben (4.2 miatt), az oszlopok a hét számának növekvő sorrendjében;
racs = racs[TELJES_HETEK]  # csak a 11 teljes hét oszlopa marad; a sorok (cikkszámok) mind megmaradnak

KONTROLL_RACS = {"Cikkszámok (sorok)":   len(abc),
                 "Hetek (oszlopok)":     len(TELJES_HETEK),
                 "Tételsorok a rácsban": szedes_abc["Het"].isin(TELJES_HETEK).sum()} # a 4.1–4.2 eredményei, a Het oszlopban szereplő azon értékek száma, amelyek a TELJES_HETEK listában szerepelnek (azaz a 15–25. hét között vannak)
mert = {"Cikkszámok (sorok)":   len(racs),
        "Hetek (oszlopok)":     len(racs.columns),
        "Tételsorok a rácsban": racs.sum().sum()} # a 4.4 eredményei, a racs táblában a cellák összege; az első .sum() oszloponként ad össze (11 szám, a heti összesen), a második ezt a 11 számot adja össze
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_RACS[nev] else "ELTÉRÉS!"
    print(f"{nev:<21} {ertek:>9,} (várt: {KONTROLL_RACS[nev]:>9,})  {allapot}")
print(f"\nCellák: {racs.size:,}, ebből 0 értékű: {(racs == 0).sum().sum() / racs.size:.1%}")
racs.head()

Cikkszámok (sorok)      269,737 (várt:   269,737)  OK
Hetek (oszlopok)             11 (várt:        11)  OK
Tételsorok a rácsban  1,943,037 (várt: 1,943,037)  OK

Cellák: 2,967,107, ebből 0 értékű: 71.5%


Het,15,16,17,18,19,20,21,22,23,24,25
Cikkszam,,,,,,,,,,,
code1000003,0,2,0,1,0,1,1,0,0,0,0
code1000024,0,0,0,0,0,0,0,1,0,1,2
code100004,0,1,0,0,0,0,1,0,0,0,1
code1000058,0,0,0,0,0,0,0,0,0,1,0
code1000075,0,0,0,0,0,0,0,0,0,0,0


### 4.5 XYZ: a relatív szórás (CV) és az osztály (D-16)

A rács minden sorára (cikkszámára) kiszámolódik a 11 heti érték átlaga és szórása, majd ezek hányadosa, a relatív szórás (CV). A szórás populációs szórás: a 11 hét a teljes vizsgált időszak, nem minta, ezért az átlagos négyzetes eltérés a hetek számával (11) osztódik. Az osztály: X, ha CV ≤ 0,5; Y, ha 0,5 < CV ≤ 1,0; Z, ha CV > 1,0. Annál a cikkszámnál, amelyet csak a csonka heteken szedtek, a rácssor csupa 0, így az átlag és a szórás is 0, a CV pedig 0/0, vagyis nem számolható (NaN). Ezek Z-be kerülnek, mert a teljes heteken egyáltalán nem volt keresletük (D-16). A kontroll: a NaN-ok száma egyezik a rács csupa 0 sorainak számával. A cella végén öt valódi cikkszám mutatja a számítást.

In [22]:
# 4.5 XYZ: a heti tételsorok relatív szórása (CV) és az osztály (D-16)
XYZ_HATAR = {"X": 0.5, "Y": 1.0}  # CV ≤ 0,5: X;  0,5 < CV ≤ 1,0: Y;  CV > 1,0: Z
KONTROLL_XYZ = {"X": 3_970, "Y": 28_740, "Z": 237_027}
PELDAK = ["code1155501", "code1000839", "code3194541", "code1003064", "code1000075"]  # AX, AY, AZ, B, csak csonka héten szedett

xyz = pd.DataFrame({"Atlag": racs.mean(axis=1), "Szoras": racs.std(axis=1, ddof=0)})  # soronként: a 11 hét átlaga és szórása; a szótárból DataFrame készül, a kulcsok ("Atlag", "Szoras") lesznek az oszlopnevek, az értékek (racs.mean(axis=1)) a sorok (a cikkszámok a sorcímkék); mean: átlag, ddof: szórás, ddof=0: a szórásnál a nevező n, nem n-1 (n-1 a pandas alapértelmezettje); axis-1: soronként számol, vízszintesen haladva
xyz["CV"] = xyz["Szoras"] / xyz["Atlag"]  # 0 / 0 = NaN: a csak csonka héten szedett cikkszámok; minden NaN → Z (a CV oszlopban a NaN értékek a Z osztályba kerülnek)
xyz["XYZ"] = "Z"
xyz.loc[xyz["CV"] <= XYZ_HATAR["Y"], "XYZ"] = "Y" # a CV ≤ 1,0: Y; a CV > 1,0: Z (már eleve Z az alapértelmezett)
xyz.loc[xyz["CV"] <= XYZ_HATAR["X"], "XYZ"] = "X" # a CV ≤ 0,5: X; 0,5 < CV <= 1,0: Y; sorrendiség itt is számít: előbb a Y, majd az X, így az X osztályba kerülnek a legkisebb CV-k, a Y-be a közepesek, a Z-be a maradék. (Ha fordítva lenne, akkor a Y-be kerülne minden, ami kisebb vagy egyenlő, mint 1,0, és az X osztály üres lenne.)
xyz = xyz.reset_index()  # a Cikkszam a sorcímkéből oszlop lesz, a sorcímke 0-tól újraszámozódik; itt már nem kell a name=..., mert a 4.2-vel ellentétben itt már tábla van, névvel ellátott oszlopokkal, nem pedig egy névtelen értéksorozat (Series); az új oszlop a sorcímkék nevét kapja: "Cikkszám"

for osztaly, vart in KONTROLL_XYZ.items(): # a már ismert kontrollértékekkel összehasonlítás
    db = (xyz["XYZ"] == osztaly).sum() # a .sum() a True értékeket 1-nek számolja, így a db a cikkszámok száma az adott osztályban
    allapot = "OK" if db == vart else "ELTÉRÉS!"
    print(f"{osztaly}: {db:>7,} cikkszám ({db / len(xyz):5.1%})  (várt: {vart:,})  {allapot}")
nan_db = xyz["CV"].isna().sum()  # hány darab NaN érték van a CV oszlopban, azaz hány cikkszám csak csonka héten szedett
nulla_sor = (racs.sum(axis=1) == 0).sum() # a rácsban hány sorban van 0 összesen
print(f"\nCV nem számolható (NaN → Z): {nan_db:,} (várt: a rács csupa 0 sorai, {nulla_sor:,})  {'OK' if nan_db == nulla_sor else 'ELTÉRÉS!'}")
xyz.loc[xyz["Cikkszam"].isin(PELDAK)]

X:   3,970 cikkszám ( 1.5%)  (várt: 3,970)  OK
Y:  28,740 cikkszám (10.7%)  (várt: 28,740)  OK
Z: 237,027 cikkszám (87.9%)  (várt: 237,027)  OK

CV nem számolható (NaN → Z): 7,043 (várt: a rács csupa 0 sorai, 7,043)  OK


,Cikkszam,Atlag,Szoras,CV,XYZ
4,code1000075,0.000000,0.000000,NaN,Z
41,code1000839,1.000000,0.953463,0.953463,Y
124,code1003064,0.454545,0.497930,1.095445,Z
7550,code1155501,1.909091,0.514259,0.269374,X
117983,code3194541,2.000000,5.443929,2.721965,Z


### 4.6 Az osztályok a DimCikk-be (D-17)

Az ABC és az XYZ a cikkszám tulajdonsága a három hónap alapján, ezért statikus oszlopként a cikktörzsbe (DimCikk) kerül, és a Power BI szűrői nem változtatják meg (D-17). Először az ABC- és az XYZ-eredmény kapcsolódik össze cikkszám szerint, majd ez bal oldali illesztéssel a DimCikk mellé kerül. Öt új oszlop jön létre: `Tetelsor`, `ABC`, `CV`, `XYZ` és a kettő együtt, `ABC_XYZ` (pl. „AX”). A 450 149 cikkszámból 180 412-nek nincs valódi szedése (179 871-et egyáltalán nem szedték, 541-nek csak 0 darabos szedése volt): ezek „Nincs szedés” kategóriát kapnak, a `Tetelsor` értékük 0, a `CV` üres marad. A kontroll: a DimCikk sorainak száma nem változik, a kulcs egyedi marad, a `Tetelsor` összege egyezik a 4.1-es tételsorok számával, és az osztályonkénti darabszámok egyeznek a 4.3-mal és a 4.5-tel.

In [23]:
# 4.6 Az osztályok a DimCikk-be, statikus oszlopként (D-17)
NINCS = "Nincs szedés"
KONTROLL_DIMCIKK = 450_149
KONTROLL_DIMCIKK_ABC = {"A": 76_209, "B": 108_934, "C": 84_594, NINCS: 180_412}

osztalyok = abc[["Cikkszam", "Tetelsor", "ABC"]].merge(xyz[["Cikkszam", "CV", "XYZ"]], how="left", on="Cikkszam", validate="one_to_one") # az "abc"-ből 3 oszlop jön (Kum_arany nem a cikk tulajdonsága, csak segédérték), az "xyz"-ből a CV és az XYZ oszlopok jönnek, a Cikkszám már megvan, a többi segédérték itt is; left join, 1:1, a Cikkszam kulccsal
DimCikk = DimCikk[CIKK_OSZLOPOK].merge(osztalyok, how="left", on="Cikkszam", validate="one_to_one")  # CIKK_OSZLOPOK: a 3.3-as "DimCikk" négy oszlopa, ehhez mergelem az osztalyok táblát; ugyan úgy LEFT JOIN, 1:1, a Cikkszam kulccsal; a DimCikk négy oszlopához jön 4 új oszlop, majd következő sorban egy ötödik. Ezzel a DimCikk megváltozott, az Export már ezt (illetve a pár sorral lejjebbi változatát) fogja betölteni a Power BI-ba, a 3.3-as DimCikk-et már nem használjuk.
DimCikk["ABC_XYZ"] = DimCikk["ABC"] + DimCikk["XYZ"]  # pl. "A" + "X" = "AX", szövegösszefűzés; a fillna előtt kell
DimCikk["Tetelsor"] = DimCikk["Tetelsor"].fillna(0).astype("int64")  # a nem szedett cikkszámoknál 0 (NaN helyett), a tételsorok száma egész szám, a Power BI-hoz int64 típus kell
for oszlop in ["ABC", "XYZ", "ABC_XYZ"]:
    DimCikk[oszlop] = DimCikk[oszlop].fillna(NINCS) # a három szöveges oszlopban a hiányzó érték helyére "Nincs szedés" kerül; a CV-t szándékosan hagyom üresen, mert a 0 azt jelentené, hogy a cikk tökéletesen egyenletesen szedett, ami nem igaz, a CV csak a szedett cikkszámokra értelmezhető

allapot = "OK" if len(DimCikk) == KONTROLL_DIMCIKK and DimCikk["Cikkszam"].is_unique else "ELTÉRÉS!" # a DimCikk-ben a sorok száma és a Cikkszam oszlop egyedisége is ellenőrzésre kerül
print(f"DimCikk: {len(DimCikk):,} sor (várt: {KONTROLL_DIMCIKK:,}), a Cikkszam egyedi: {DimCikk['Cikkszam'].is_unique}  {allapot}")
osszeg = DimCikk["Tetelsor"].sum() # minden szedés megérkezett a DimCikk-be, egy sem veszett el, egy sem duplázódott, a nem szedettek 0-val járulnak hozzá
print(f"Tételsorok a DimCikk-ben: {osszeg:,} (várt: {KONTROLL_ABC_SOR:,})  {'OK' if osszeg == KONTROLL_ABC_SOR else 'ELTÉRÉS!'}\n")
for osztaly, vart in KONTROLL_DIMCIKK_ABC.items():
    db = (DimCikk["ABC"] == osztaly).sum()
    print(f"ABC {osztaly:<13} {db:>8,} (várt: {vart:>8,})  {'OK' if db == vart else 'ELTÉRÉS!'}")
for osztaly, vart in KONTROLL_XYZ.items():
    db = (DimCikk["XYZ"] == osztaly).sum()
    print(f"XYZ {osztaly:<13} {db:>8,} (várt: {vart:>8,})  {'OK' if db == vart else 'ELTÉRÉS!'}")
print()
print(DimCikk.dtypes)
DimCikk.head()

DimCikk: 450,149 sor (várt: 450,149), a Cikkszam egyedi: True  OK
Tételsorok a DimCikk-ben: 2,095,722 (várt: 2,095,722)  OK

ABC A               76,209 (várt:   76,209)  OK
ABC B              108,934 (várt:  108,934)  OK
ABC C               84,594 (várt:   84,594)  OK
ABC Nincs szedés   180,412 (várt:  180,412)  OK
XYZ X                3,970 (várt:    3,970)  OK
XYZ Y               28,740 (várt:   28,740)  OK
XYZ Z              237,027 (várt:  237,027)  OK

Cikkszam                   str
Termekcsoportkod           str
Termekcsoportkodnev        str
Gyartokod                  str
Tetelsor                 int64
ABC                        str
CV                     float64
XYZ                        str
ABC_XYZ                    str
dtype: object


,Cikkszam,Termekcsoportkod,Termekcsoportkodnev,Gyartokod,Tetelsor,ABC,CV,XYZ,ABC_XYZ
0,code912814,377,Forgattyúsház fedél,H78,14,A,0.870285,Y,AY
1,code244300,429,Gázos (Cofap) lengéscsillapító,P94,423,A,0.222020,X,AX
2,code3758861,286,Üzemanyagnyomás-szabályzó,I98,9,A,1.143959,Z,AZ
3,code4640689,556,Rozsdaoldó - jövedéki,020,34,A,0.724743,Y,AY
4,code202494,1244,Tartófal,L64,225,A,0.342808,X,AX


### 4.7 Az ABC–XYZ mátrix

A két besorolás együtt kilenc csoportot ad (AX, AY, AZ, …, CZ). A mátrix sorai az ABC-, oszlopai az XYZ-osztályok. Az első tábla cellánként a cikkszámok számát, a második a szedési tételsorok arányát mutatja, összesen sorral és oszloppal. A nem szedett cikkszámok kimaradnak. A kontroll: a cikkszámok összesen egyezik a szedett cikkszámok számával (4.2), a tételsorok összesen pedig a 4.1-es tételsorok számával. A Power BI-ban ugyanez a mátrix a DimCikk `ABC` és `XYZ` oszlopából készül, és a számainak ezekkel kell egyezniük.

In [24]:
# 4.7 ABC–XYZ mátrix: a cikkszámok száma és a tételsorok aránya cellánként
szedett_cikk = DimCikk.loc[DimCikk["ABC"] != NINCS]  # csak a szedett cikkszámok sorai maradnak

matrix_cikk = pd.crosstab(szedett_cikk["ABC"], szedett_cikk["XYZ"], margins=True, margins_name="Összesen")  # a sorok az ABC osztályok (A, B, C), az oszlopok az XYZ osztályok (X, Y, Z), a cellákban a cikkszámok száma; a margins=True a sorok és oszlopok összegét is kiszámolja (egy új sort és egy új oszlopot ad hozzá), az Összesen névvel; amit mutat: hány cikkszám tartozik egy adott csoportba (pl. hány cikkszám van az AX cellában(3970))
matrix_sor = pd.crosstab(szedett_cikk["ABC"], szedett_cikk["XYZ"], values=szedett_cikk["Tetelsor"], aggfunc="sum", # a values=szedett_cikk["Tetelsor"], aggfunc="sum": ezzel a crosstab() nem a sorokat számolja, hanem cellánként összeadja a Tetelsor értékeit (az A × X cella például a 3970 AX cikkszám tételsorainak összege lesz); amit mutat: hányszor szedték összesen ezeket a cikkszámokat (pl. AX cellában a 3970 cikkszámnak 401 128 szedésük volt)
                         margins=True, margins_name="Összesen").fillna(0).astype("int64")  # tételsorok összege; a BX, BY, CX, CY cellákban egyetlen cikkszám sincs, így nincs mit összeadni, a NaN eredményt a fillna(0) 0-ra cseréli; a NaN miatt az oszlopok tört típusúak lettek, a .astype("int64") visszaalakítja őket egész számmá; ez is kapja az "Összesen" sort és oszlopot; a matrix_sor köztes tábla, a matrix_aranyhoz kell
matrix_arany = (matrix_sor / matrix_sor.loc["Összesen", "Összesen"] * 100).round(1)  # a tételsorok aránya, %; a két "Összesen" cella a teljes tételsor (egyetlen cella: a jobb alsó), így minden cellát ezzel osztunk, majd 100-zal szorzunk, és a .round(1) egy tizedesjegyre kerekít

KONTROLL_MATRIX = {"Cikkszámok összesen": len(abc), "Tételsorok összesen": KONTROLL_ABC_SOR}  # 4.2 és 4.1
mert = {"Cikkszámok összesen": matrix_cikk.loc["Összesen", "Összesen"], # a crosstab() a sorok és oszlopok összegét is kiszámolja, az Összesen névvel; a .loc a sorcímke alapján keresi az "Összesen" sort, majd az oszlopcímke alapján az "Összesen" oszlopot, így a jobb alsó cellát adja vissza
        "Tételsorok összesen": matrix_sor.loc["Összesen", "Összesen"]}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_MATRIX[nev] else "ELTÉRÉS!"
    print(f"{nev}: {ertek:>9,} (várt: {KONTROLL_MATRIX[nev]:>9,})  {allapot}")

print("\nCikkszámok száma:")
print(matrix_cikk)
print("\nA szedési tételsorok aránya (%):")
print(matrix_arany)

Cikkszámok összesen:   269,737 (várt:   269,737)  OK
Tételsorok összesen: 2,095,722 (várt: 2,095,722)  OK

Cikkszámok száma:
XYZ          X      Y       Z  Összesen
ABC                                    
A         3970  28740   43499     76209
B            0      0  108934    108934
C            0      0   84594     84594
Összesen  3970  28740  237027    269737

A szedési tételsorok aránya (%):
XYZ          X     Y     Z  Összesen
ABC                                 
A         19.1  39.5  21.7      80.3
B          0.0   0.0  15.6      15.6
C          0.0   0.0   4.0       4.0
Összesen  19.1  39.5  41.4     100.0


### 4.8 A készlet ABC-osztályonként

A FactKeszlet és a DimCikk a cikkszámon keresztül kapcsolódik (3. szakasz), így a tárhelyeken lévő készlet is bontható a szedési ABC-osztály szerint. A cella megmutatja, hogy a tárhely × cikkszám párok és a készletdarabok hány százaléka tartozik az A, B, C és a „Nincs szedés” cikkszámokhoz. A FactKeszlet a legfrissebb állapot pillanatképe (D-14). A kontroll: a párok és a darabok összege egyezik a 3.5-ös kontrollszámokkal, és minden tárhelysor kapott osztályt.

In [25]:
# 4.8 A készlet ABC-osztályonként: tárhely × cikkszám párok és készletdarab (D-14, D-17)
keszlet_abc = FactKeszlet.merge(DimCikk[["Cikkszam", "ABC"]], how="left", on="Cikkszam", validate="many_to_one")  # a tárhelysorok mellé az ABC: a FactKeszlethez a DimCikk ABC oszlopa jön

keszlet_osszesito = pd.DataFrame({
    "Tarhelypar": keszlet_abc.groupby("ABC").size(),             # osztályonként megszámolja a tárhelysorokat, azaz a tárhely × cikkszám párok száma; a size() olyan mint az SQL COUNT(*) (ezárt sem kell neki oszlopot megadni)
    "Darab":      keszlet_abc.groupby("ABC")["Darab"].sum(),     # osztályonként a készletdarab; a Darabot kell összesíteni, nem a sorok számát, így a ["Darab"] oszlopot kell kiválasztani, majd a .sum() összegzi az osztályonkénti Darab értékeket
})
keszlet_osszesito["Tarhelypar_%"] = (keszlet_osszesito["Tarhelypar"] / keszlet_osszesito["Tarhelypar"].sum() * 100).round(1) # a tarhelypar aránya, %; az osztályonkénti tárhelypar / összes tárhelypar * 100, kerekítve egy tizedesjegyre
keszlet_osszesito["Darab_%"] = (keszlet_osszesito["Darab"] / keszlet_osszesito["Darab"].sum() * 100).round(1) # a darab aránya, %; az osztályonkénti darab / összes darab * 100, kerekítve egy tizedesjegyre

mert = {"Sorok": keszlet_osszesito["Tarhelypar"].sum(), "Készletdarab": keszlet_osszesito["Darab"].sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_KESZLET[nev] else "ELTÉRÉS!"
    print(f"{nev:<13} {ertek:>10,} (várt: {KONTROLL_KESZLET[nev]:>10,})  {allapot}")
print(f"Osztály nélküli tárhelysor: {keszlet_abc['ABC'].isna().sum()}\n")
keszlet_osszesito

Sorok            834,925 (várt:    834,925)  OK
Készletdarab   5,401,239 (várt:  5,401,239)  OK
Osztály nélküli tárhelysor: 0



,Tarhelypar,Darab,Tarhelypar_%,Darab_%
ABC,,,,
A,325114,3862909,38.9,71.5
B,175807,663745,21.1,12.3
C,98476,263078,11.8,4.9
Nincs szedés,235528,611507,28.2,11.3


### 4.9 Az ABC–XYZ eredmény értelmezése

**A mátrix (4.7).** A 269 737 szedett cikkszám így oszlik meg:
- **AX:** 3 970 cikkszám (1,5%), a szedések 19,1%-a. Gyakran és hétről hétre egyenletesen szedik.
- **AY:** 28 740 cikkszám (10,7%), a szedések 39,5%-a. Gyakran, mérsékelt ingadozással.
- **AZ:** 43 499 cikkszám (16,1%), a szedések 21,7%-a. Gyakran, de hetente erősen változó számban (pl. kilenc hét szedés nélkül, majd egy héten 19 szedés).
- **BZ és CZ:** 193 528 cikkszám (71,7%), együtt a szedések 19,6%-a. Ritkán, szórványosan szedik. Hogy minden B és C cikkszám Z, az a módszer következménye: legfeljebb öt szedés a 11 héten nem lehet egyenletes (4.5).

Az AX és az AY együtt 32 710 cikkszám (12,1%), és a szedések 58,6%-át adja.

**Mit jelent ez a raktár számára (javaslat):** A szedés munkája tételsoronként jelentkezik (a tárhelyhez menni, kivenni, lekönyvelni), ezért a gyakran szedett cikkszámok elhelyezése határozza meg a munka nagy részét.
- **Tárhelykiosztás:** az A-cikkszámokat (a cikkszámok 28,3%-a, a szedések 80,3%-a) a legkönnyebben és leggyorsabban elérhető tárhelyekre érdemes tenni; a B- és C-cikkszámok távolabbi helyen is lehetnek, mert együtt a szedések 19,6%-át adják.
- **Utánpótlás** (a szedési tárhely feltöltése a tartalékkészletből): az X- és Y-cikkszámok heti igénye kiszámítható, ezért a feltöltésük rendszeres ütemben tervezhető. Az AZ-cikkszámoknál a heti igény ugrálhat, ezért a szedési helyen nagyobb tartalék vagy gyakoribb ellenőrzés kell.

**A készlet (4.8):** Az A-cikkszámok a tárhely × cikkszám párok 38,9%-án a készletdarab 71,5%-át tartják. A „Nincs szedés” cikkszámok a párok 28,2%-át foglalják (235 528 pár, a készletdarab 11,3%-a): ezeket a három hónap alatt egyszer sem szedték. Ez felülvizsgálati lista, nem selejtezési javaslat: három hónap rövid időszak, és a pótalkatrészek között lehetnek szezonális vagy ritkán, de rendszeresen keresett tételek.

**Korlátok:**
- Az ABC a szedések számán alapul, nem értéken, mert az adatokban nincs ár (D-15); egy ritkán szedett, de drága cikkszám itt C.
- Három hónap adat: szezonális hatás nem látszik.
- A heti bontás és a sok kis forgalmú cikkszám miatt a Z aránya magas; az XYZ az A-osztályon belül különböztet igazán.
- A besorolás statikus, a 2026. április–júniusi időszakot írja le (D-17).

## 5. Teljesítménypont

A feladat olyan kimutatást kér, amely bemutatja, hogy a csapat tagjai a létszámalapú teljesítményszámítás alapján hány teljesítménypontot szereztek. A pontot a forrásrendszer adja: minden betárolási és szedési sor 1 vagy 2 pontot kapott. A pont nem függ a sor darabszámától (5.1), ezért egy dolgozó pontszáma az elvégzett tételsorai számát követi, nem a mozgatott mennyiséget. A notebook nem számol pontot, csak összeadja a soronkénti értékeket.

**Kik szerepelnek:** a Létszám tábla 27 dolgozója, vagyis a Csoport 1 (12 fő) és a Csoport 2 (15 fő). A csoporton kívüli kódok és a gépi könyvelés („Rendszer”) sorai kimaradnak (D-07, D-08).

**Döntések:**
- **Két nézet:** dolgozónként, „Név (kód)” formában, és csoportonként egy főre. Az egy főre jutó pont a csoport összes pontja osztva a csoport létszámával (12, illetve 15 fő). A két csoport létszáma eltér, ezért az összpontjuk közvetlenül nem vethető össze, az egy főre jutó pontjuk igen (D-09, D-18).
- **Betárolás és szedés együtt és külön:** a két folyamat pontja azonos skálán van (soronként 1 vagy 2), ezért összeadható. A folyamatok szerinti bontás a két csoport munkamegosztását mutatja.
- **A 0 darabos szedési sorok pontjai bent maradnak:** a rendszer ezekre a sorokra is adott pontot, a feladat pedig a megszerzett pontokat kérdezi. A kizárásuk hatását az 5.3 kiírja (A-07, D-06).
- **Havi bontás** csoportonként, a lezárás napja szerint (D-03). A június csak 23 napot tartalmaz (06.23-ig), ezért a júniusi pontszám eleve kisebb.

**Korlát:** az adatban nincs munkaidő, ezért a pontszám a könyvelt munka mennyiségét mutatja, a hatékonyságot nem.

**A szakasz lépései:** 5.1 a két csoport mozgássorai; 5.2 pontok dolgozónként; 5.3 csoportonként egy főre; 5.4 havi bontás; 5.5 az eredmény értelmezése.

### 5.1 A két csoport mozgássorai (D-07, D-08)

A FactMozgas minden sora mellé bal oldali illesztéssel a dolgozó neve és csoportja (DimDolgozo), valamint a hónap (DimDatum) kerül. A teljesítménypont és a KPI csak a két csoport sorait használja, ezért a további lépések egy szűrt táblából dolgoznak (`csoport_mozgas`), amelyben csak a szükséges oszlopok maradnak; a 6. szakasz is ebből számol. A kontrollok: a négy kategória (Csoport 1, Csoport 2, Csoporton kívüli, Rendszer) pontjainak összege egyezik a tisztított táblák pontösszegével (2.7); a két csoport sorainak száma egyezik a Létszám tábla kódjaihoz tartozó mozgássorok számával, amelyet a cella a csoportbesorolás nélkül, közvetlenül a FactMozgas-ból számol; a két csoportban 27 különböző dolgozó van; és minden sornak van hónapja. A cella végén a két csoport sorai folyamat és pontérték szerint: ha a pont a darabtól függne, a 2 pontos sorok átlagos darabszáma nagyobb lenne.

In [26]:
# 5.1 A két csoport mozgássorai a dolgozó, a csoport és a hónap adataival (D-07, D-08)
CSOPORTOK = ["Csoport 1", "Csoport 2"]
KONTROLL_PONT = KONTROLL_TISZTA["Betárolási teljesítménypont"] + KONTROLL_TISZTA["Szedési teljesítménypont"]  # a 2.7-es kontrollszótárból kiveszi a betárolási (1 225 119) és a szedési (3 211 882) teljesítménypontot, és összeadja őket: 4 437 001. Ez a tisztított adatok összes pontja, minden ügyintézővel együtt
KONTROLL_CSOPORT_SOR = FactMozgas["ugyintezo"].isin(letszam["ugyintezo_kod"]).sum()  # független út; a letszam az 1.1-ben betöltött eredeti Létszám tábla, 27 kóddal; az .isin a FactMozgas ugyintezo oszlopminden sorára megnézi, hogy a kód szerepel-e a 27 között ami a letszam táblában van, a .sum() pedig megszámolja a True értékeket

mozgas = FactMozgas.merge(DimDolgozo[["ugyintezo", "Nev_kod", "Csoport"]], how="left", on="ugyintezo", validate="many_to_one")  # minden mozgássor mellé a dolgozó neve és csoportja (3.2)
mozgas = mozgas.merge(DimDatum[["Datum", "Honap"]], how="left", on="Datum", validate="many_to_one")  # majd a hónap is a naptártáblából (3.4)

kategoriak = pd.DataFrame({  # új DataFrame készül, a kulcsok (Sorok, Pont) lesznek az oszlopnevek, az értékek a sorozatok (Series) lesznek, a sorcímkék a csoportok nevei; kontrollhoz kell
    "Sorok": mozgas.groupby("Csoport").size(),                     # sorok száma a négy kategóriában (COUNT(*))
    "Pont":  mozgas.groupby("Csoport")["teljesitmenypont"].sum(),  # pontok összegea négy kategóriában (SUM(teljesitmenypont))
})
csoport_mozgas = mozgas.loc[mozgas["Csoport"].isin(CSOPORTOK), # True, ha a Csoport oszlopban a két csoport egyikének neve szerepel, False, ha nem; a .loc a True értékek sorait adja vissza, a szögletes zárójelben a listában szereplő oszlopnev(ek) maradnak meg
                            ["Csoport", "ugyintezo", "Nev_kod", "statusz", "Honap", "darab", "teljesitmenypont", "nulla_darab"]]  # csak a két csoport sorai, a sok felsorolt oszlop is bekerül a csoport_mozgas táblába, a többi oszlopot eldobja

KONTROLL_51 = {"Pont a négy kategóriában":  KONTROLL_PONT,
               "A két csoport sorai":       KONTROLL_CSOPORT_SOR,
               "Dolgozók a két csoportban": len(letszam),
               "Hónap nélküli sorok":       0}
mert = {"Pont a négy kategóriában":  kategoriak["Pont"].sum(),
        "A két csoport sorai":       len(csoport_mozgas),
        "Dolgozók a két csoportban": csoport_mozgas["ugyintezo"].nunique(),
        "Hónap nélküli sorok":       csoport_mozgas["Honap"].isna().sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_51[nev] else "ELTÉRÉS!"
    print(f"{nev:<26} {ertek:>10,} (várt: {KONTROLL_51[nev]:>10,})  {allapot}")

print("\nSorok és pontok a négy kategóriában:")
print(kategoriak)

pont_ertek = pd.DataFrame({
    "Sorok":         csoport_mozgas.groupby(["statusz", "teljesitmenypont"]).size(), # eddig a .groupby egy oszlopot kapott, itt egy listát kap, két oszlopnévvel; a csoportokat ilyenkor a két oszlop értékpárjai adják: ahány különböző (statusz, teljesitmenypont) pár van, annyi csoport. Itt 2 × 2 = 4
    "Átlagos darab": csoport_mozgas.groupby(["statusz", "teljesitmenypont"])["darab"].mean().round(2),  # a sorok átlagos darabszáma ugyanígy bontva, 2 tizedesjegyre kerekítve
})
print("\nA két csoport sorai pontérték szerint:")
print(pont_ertek)

Pont a négy kategóriában    4,437,001 (várt:  4,437,001)  OK
A két csoport sorai           568,343 (várt:    568,343)  OK
Dolgozók a két csoportban          27 (várt:         27)  OK
Hónap nélküli sorok                 0 (várt:          0)  OK

Sorok és pontok a négy kategóriában:
                    Sorok     Pont
Csoport                           
Csoport 1          251026   376976
Csoport 2          317317   476120
Csoporton kívüli  2166123  3249176
Rendszer           223306   334729

A két csoport sorai pontérték szerint:
                             Sorok  Átlagos darab
statusz   teljesitmenypont                       
BETAROLAS 1                  75082           3.32
          2                  75590           3.22
SZEDES    1                 208508           1.51
          2                 209163           1.48


### 5.2 Pontok dolgozónként (D-09, D-18)

A két csoport sorai dolgozónként és folyamatonként összegződnek: egy sor egy dolgozó, a betárolási, a szedési és az összes pontjával. A dolgozót az ügyintézőkód azonosítja; a megjelenítéshez a „Név (kód)” oszlop tartozik, mert két-két dolgozónak azonos a neve (D-09). A tábla az összes pont szerint csökkenő sorrendben áll: ez a dolgozói rangsor (D-18). A kontroll: a táblának 27 sora van (a Létszám tábla dolgozói), és a pontok összege egyezik a két csoport 5.1-es pontjával. A pontszám a könyvelt munka mennyiségét mutatja, munkaidő nélkül (5.0).

In [27]:
# 5.2 Pontok dolgozónként: betárolás, szedés és összesen; rangsor az összes pont szerint (D-09, D-18)
KONTROLL_52 = {"Dolgozók": len(letszam), "Pont összesen": kategoriak.loc[CSOPORTOK, "Pont"].sum()}  # len(letszam) = 27, az eredeti Létszám tábla sorainak száma (1.1); az 5.1-es kategoriak táblából a „Csoport 1” és a „Csoport 2” sor Pont értéke (376 976 + 476 120 = 853 096); a .loc első helyén most nem egy sorcímke vagy maszk áll, hanem egy sorcímke-lista: a lista minden eleméhez kiveszi a sort (5.1)

pont_dolgozo = csoport_mozgas.pivot_table(index=["Csoport", "ugyintezo", "Nev_kod"], columns="statusz", # a pivot_table() a csoport_mozgas táblából készít egy új táblát, ahol a sorok a három index oszlop értékei (egyedi kombinációk), az oszlopok a statusz oszlop értékei (BETAROLAS, SZEDES), a cellákban pedig a teljesitmenypont oszlop értékeinek összege lesz (egy dolgozónak hány teljesítménypontja van betárolásból és szedésből); a pivot_table() alapértelmezettje az aggfunc="mean", de itt explicit módon megadjuk, hogy az összeget számolja; a fill_value=0 a hiányzó értékeket NaN-ról 0-ra cseréli, így ha egy dolgozónak nincs pontja az egyik folyamatból, akkor 0 lesz a cellában
                                          values="teljesitmenypont", aggfunc="sum", fill_value=0).reset_index()  # sor: dolgozó, oszlop: folyamat, cella: a pontok összege; Miért három oszlop az index-ben, ha egy dolgozó egy sor? Mert mindhármat meg akarjuk tartani az eredményben; egy kódhoz pontosan egy Nev_kod és egy Csoport tartozik, ezért a hármas nem bontja tovább a dolgozót
pont_dolgozo = pont_dolgozo.rename(columns={"BETAROLAS": "Pont_betarolas", "SZEDES": "Pont_szedes"})  # beszédes oszlopnevek
pont_dolgozo.columns.name = None  # az oszlopok tengelyének neve („statusz”) a pivotból maradt; a kész táblában nincs szerepe
pont_dolgozo["Pont_osszes"] = pont_dolgozo["Pont_betarolas"] + pont_dolgozo["Pont_szedes"]  # az előbb megadott 2 új oszlopnévvel ezen két oszlop összege kerül az új oszlopba, a teljes pontszám dolgozónként
pont_dolgozo = pont_dolgozo.sort_values("Pont_osszes", ascending=False, ignore_index=True)  # rendezés: Pont_osszes szerint csökkenő sorrendben, azonos pontszámnál a sorok sorrendje nem számít; az új sorszámok 0-tól indulnak, a 0 a legtöbb pontot szerzett dolgozó

mert = {"Dolgozók": len(pont_dolgozo), "Pont összesen": pont_dolgozo["Pont_osszes"].sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_52[nev] else "ELTÉRÉS!"
    print(f"{nev:<14} {ertek:>9,} (várt: {KONTROLL_52[nev]:>9,})  {allapot}")
pont_dolgozo

Dolgozók              27 (várt:        27)  OK
Pont összesen    853,096 (várt:   853,096)  OK


,Csoport,ugyintezo,Nev_kod,Pont_betarolas,Pont_szedes,Pont_osszes
0,Csoport 2,8759,Nagy Anna (8759),5406,39203,44609
1,Csoport 1,8733,Kovács Gábor (8733),3165,40947,44112
2,Csoport 2,7033,Szabó Erika (7033),9551,33519,43070
3,Csoport 2,9606,Nagy Katalin (9606),21473,19572,41045
4,Csoport 1,6272,Varga Mária (6272),3225,37366,40591
5,Csoport 2,7491,Molnár Hanna (7491),17562,21560,39122
6,Csoport 2,6156,Farkas Béla (6156),16415,20723,37138
7,Csoport 1,6155,Németh Ferenc (6155),7470,28902,36372
8,Csoport 1,5176,Németh Erika (5176),8712,25026,33738
9,Csoport 2,6015,Tóth Erika (6015),15159,17882,33041


### 5.3 Pontok csoportonként és egy főre (D-09, D-18, A-07)

A dolgozói pontok (5.2) csoportonként összeadódnak, és a csoport létszámával osztva adják az egy főre jutó pontot, a betárolásra, a szedésre és a kettőre együtt. A létszám a dolgozói dimenzióból jön (12 és 15 fő, D-09); a cella ellenőrzi, hogy a csoportok minden tagjának van pontja, vagyis az aktív dolgozók száma egyezik a létszámmal. Az egy főre jutó pont teszi összevethetővé a két eltérő létszámú csoportot (D-18). A cella kiírja azt is, mennyi pontot adnak a 0 darabos szedési sorok, és mennyi lenne az egy főre jutó pont nélkülük (A-07): a döntés szerint ezek a pontok bent maradnak, a kiírás csak a hatásukat mutatja.

In [28]:
# 5.3 Pontok csoportonként és egy főre; a 0 darabos sorok pontjainak hatása (D-09, D-18, A-07)
LETSZAM = DimDolgozo.loc[DimDolgozo["Csoport"].isin(CSOPORTOK)].groupby("Csoport").size()  # a két csoport létszáma a dolgozói dimenzióból (3.2, D-09)

pont_csoport = pont_dolgozo.groupby("Csoport")[["Pont_betarolas", "Pont_szedes", "Pont_osszes"]].sum()  # a dolgozói pontok (5.2) csoportonként összeadva; a .sum() mindhárom oszlopra érvényes, így a három oszlop összege kerül a pont_csoport táblába; a sorcímkék a csoportok nevei lesznek, az oszlopnevek a három ponttípus (SELECT Csoport, SUM(Pont_betarolas), SUM(Pont_szedes), SUM(Pont_osszes) FROM pont_dolgozo GROUP BY Csoport)
pont_csoport["Letszam"] = LETSZAM                                     # egy sorozat oszlopként kerül a táblába, és a pandas a sorcímke szerint illeszti: a „Csoport 1” sorba a „Csoport 1” értéke (12) kerül
pont_csoport["Aktiv_dolgozo"] = pont_dolgozo.groupby("Csoport").size()  # megszámolja, hány dolgozói sor van csoportonként az 5.2-es táblában; az 5.2 csak azokat a dolgozókat tartalmazza, akiknek volt mozgássoruk, így ez az aktív dolgozók száma; ezt veti össze a kontroll a létszámmal (A.5)
pont_csoport["Betarolas_per_fo"] = pont_csoport["Pont_betarolas"] / pont_csoport["Letszam"] # soronkénti osztások
pont_csoport["Szedes_per_fo"]    = pont_csoport["Pont_szedes"] / pont_csoport["Letszam"]
pont_csoport["Pont_per_fo"]      = pont_csoport["Pont_osszes"] / pont_csoport["Letszam"]

nulla_pont = csoport_mozgas.loc[csoport_mozgas["nulla_darab"]].groupby("Csoport")["teljesitmenypont"].sum()  # a 0 darabos sorok pontjai csoportonként (A-07) (csoport_mozgas az 5.1-ből jön)
per_fo_nelkule = (pont_csoport["Pont_osszes"] - nulla_pont) / pont_csoport["Letszam"]                       # az egy főre jutó pont, ha ezek a 0 darabos pontok kimaradnának

KONTROLL_53 = {"Létszám, Csoport 1": KONTROLL_CSOPORT["Csoport 1"], "Létszám, Csoport 2": KONTROLL_CSOPORT["Csoport 2"],
               "Pont összesen": KONTROLL_52["Pont összesen"]}  # a 3.2 és az 5.2 kontrollszámai
mert = {"Létszám, Csoport 1": LETSZAM["Csoport 1"], "Létszám, Csoport 2": LETSZAM["Csoport 2"],
        "Pont összesen": pont_csoport["Pont_osszes"].sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_53[nev] else "ELTÉRÉS!"
    print(f"{nev:<19} {ertek:>9,} (várt: {KONTROLL_53[nev]:>9,})  {allapot}")
aktiv_ok = (pont_csoport["Aktiv_dolgozo"] == pont_csoport["Letszam"]).all()
print(f"Minden csoporttagnak van pontja (aktív = létszám): {'OK' if aktiv_ok else 'ELTÉRÉS!'}\n")

for csoport in CSOPORTOK:
    print(f"{csoport}: a 0 darabos sorok pontja {nulla_pont[csoport]:,} ({nulla_pont[csoport] / pont_csoport.loc[csoport, 'Pont_osszes']:.1%}); "
          f"egy főre {pont_csoport.loc[csoport, 'Pont_per_fo']:,.1f}, nélkülük {per_fo_nelkule[csoport]:,.1f} pont")

pont_csoport = pont_csoport.reset_index()  # a Csoport a sorcímkéből oszlop lesz (az exporthoz)
pont_csoport.round(1)

Létszám, Csoport 1         12 (várt:        12)  OK
Létszám, Csoport 2         15 (várt:        15)  OK
Pont összesen         853,096 (várt:   853,096)  OK
Minden csoporttagnak van pontja (aktív = létszám): OK

Csoport 1: a 0 darabos sorok pontja 6,256 (1.7%); egy főre 31,414.7, nélkülük 30,893.3 pont
Csoport 2: a 0 darabos sorok pontja 6,145 (1.3%); egy főre 31,741.3, nélkülük 31,331.7 pont


,Csoport,Pont_betarolas,Pont_szedes,Pont_osszes,Letszam,Aktiv_dolgozo,Betarolas_per_fo,Szedes_per_fo,Pont_per_fo
0,Csoport 1,63934,313042,376976,12,12,5327.8,26086.8,31414.7
1,Csoport 2,162328,313792,476120,15,15,10821.9,20919.5,31741.3


### 5.4 Havi bontás csoportonként (D-03, D-18)

A két csoport pontjai hónaponként és folyamatonként összegződnek; a hónapot a lezárás napja adja (D-03). Az egy főre jutó havi pont a havi összes pont osztva a csoport létszámával, így a két csoport havonta is összevethető (D-18). A cella megszámolja, hány különböző dolgozónak volt sora az adott hónapban: ha ez minden hónapban egyezik a létszámmal, a létszám mint nevező minden hónapra helyes. A június csak 06.23-ig tart (23 nap), ezért a júniusi pont eleve kisebb; a hónapok összevetésénél ezt figyelembe kell venni. A kontroll: 6 sor (2 csoport × 3 hónap); a pontok összege egyezik az 5.2-ével; csoportonként a három hónap összege egyezik az 5.3 csoportösszegével.

In [29]:
# 5.4 Havi bontás csoportonként: pontok és egy főre jutó pont havonta (D-03, D-18)
KONTROLL_54 = {"Sorok (2 csoport × 3 hónap)": len(CSOPORTOK) * 3, "Pont összesen": KONTROLL_52["Pont összesen"]}  # A len(CSOPORTOK) * 3 = 2 × 3 = 6: ennyi sor kell, ha mindkét csoportnak mindhárom hónapban van sora; az pontösszeg az 5.2-es kontrollszám (853096)

pont_havi = csoport_mozgas.pivot_table(index=["Csoport", "Honap"], columns="statusz", values="teljesitmenypont", # az index most a Csoport és a Honap: minden (csoport, hónap) pár egy sor lesz, 2 × 3 = 6 sor; nyomkövetés egy cellán: a Csoport 1 áprilisi szedési sorainak pontjai összesen 105 277, ez kerül a (Csoport 1, 4) sor SZEDES oszlopába
                                       aggfunc="sum", fill_value=0).reset_index()  # sor: csoport × hónap, oszlop: folyamat, cella: a pontok összege; fill_value=0 a hiányzó értékeket NaN-ról 0-ra cseréli, így ha egy csoportnak nincs pontja az egyik folyamatból, akkor 0 lesz a cellában; a reset_index() a Csoport és Honap oszlopokat visszahelyezi az indexből az oszlopok közé
pont_havi = pont_havi.rename(columns={"BETAROLAS": "Pont_betarolas", "SZEDES": "Pont_szedes"}) # oszlopok átnevezése beszédesebbre;
pont_havi.columns.name = None # az oszlopok tengelyének neve („statusz”) a pivotból maradt; a kész táblában nincs szerepe
pont_havi["Pont_osszes"] = pont_havi["Pont_betarolas"] + pont_havi["Pont_szedes"] # az előbb megadott 2 új oszlopnévvel ezen két oszlop összege kerül az új oszlopba, a teljes pontszám csoportonként és hónaponként

aktiv = csoport_mozgas.groupby(["Csoport", "Honap"])["ugyintezo"].nunique().reset_index(name="Aktiv_dolgozo")  # hány különböző dolgozónak volt sora az adott hónapban; csoport_mozgas.groupby(["Csoport", "Honap"]): 6 csoport, például (Csoport 1, 4) = a Csoport 1 áprilisi sorai; ...["ugyintezo"]: csoportonként az ügyintézőkódok oszlopa; ...nunique(): csoportonként a különböző kódok száma; (Csoport 1, 4) → 12 itt csoportonként fut (mivel egy groupby után jön), nem a sorokat számolja (abból a Csoport 1-nél áprilisban több ezer van), hanem azt, hány különböző dolgozó könyvelt; ...reset_index(name="Aktiv_dolgozo"): táblává alakítja: Csoport, Honap, Aktiv_dolgozo oszlop, 6 sor (mint a 4.2-ben)
pont_havi = pont_havi.merge(aktiv, how="left", on=["Csoport", "Honap"], validate="one_to_one")  # illesztés két kulccsal: csoport és hónap
pont_havi["Letszam"] = pont_havi["Csoport"].map(LETSZAM)       # a csoport létszáma; a .map az 5.3 LETSZAM sorozatot (mint korábban 3.4-ben a szótárat is) használja (Csoport 1 → 12, Csoport 2 → 15); a .map ugyanúgy használja: a sorcímke a kulcs, az érték az érték; a „Csoport 1” sorok 12-t, a „Csoport 2” sorok 15-öt kapnak
pont_havi["Pont_per_fo"] = pont_havi["Pont_osszes"] / pont_havi["Letszam"]
pont_havi["HonapNev"] = pont_havi["Honap"].map(HONAPNEVEK)     # a hónap neve (3.4-es szótár)

mert = {"Sorok (2 csoport × 3 hónap)": len(pont_havi), "Pont összesen": pont_havi["Pont_osszes"].sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_54[nev] else "ELTÉRÉS!"
    print(f"{nev:<28} {ertek:>9,} (várt: {KONTROLL_54[nev]:>9,})  {allapot}")
havi_osszeg = pont_havi.groupby("Csoport")["Pont_osszes"].sum()                                # csoportonként a három hónap összege
csoport_ok = (pont_csoport["Csoport"].map(havi_osszeg) == pont_csoport["Pont_osszes"]).all()  # egyezik-e az 5.3 csoportösszegével
aktiv_ok = (pont_havi["Aktiv_dolgozo"] == pont_havi["Letszam"]).all()                         # minden hónapban minden csoporttag dolgozott-e
print(f"A havi pontok csoportonként = az 5.3 csoportösszege: {'OK' if csoport_ok else 'ELTÉRÉS!'}")
print(f"Minden hónapban minden csoporttagnak van sora:       {'OK' if aktiv_ok else 'ELTÉRÉS!'}")
pont_havi.round(1)

Sorok (2 csoport × 3 hónap)          6 (várt:         6)  OK
Pont összesen                  853,096 (várt:   853,096)  OK
A havi pontok csoportonként = az 5.3 csoportösszege: OK
Minden hónapban minden csoporttagnak van sora:       OK


,Csoport,Honap,Pont_betarolas,Pont_szedes,Pont_osszes,Aktiv_dolgozo,Letszam,Pont_per_fo,HonapNev
0,Csoport 1,4,21485,105277,126762,12,12,10563.5,április
1,Csoport 1,5,23947,106691,130638,12,12,10886.5,május
2,Csoport 1,6,18502,101074,119576,12,12,9964.7,június
3,Csoport 2,4,52342,105169,157511,15,15,10500.7,április
4,Csoport 2,5,58755,113773,172528,15,15,11501.9,május
5,Csoport 2,6,51231,94850,146081,15,15,9738.7,június


### 5.5 A teljesítménypont értelmezése

- **A pont mérőszáma:** a forrásrendszer soronként 1 vagy 2 pontot ad, a darabtól függetlenül (5.1), ezért a pontszám az elvégzett tételsorok számát követi, nem a mozgatott mennyiséget.
- **Egy főre a két csoport szinte azonos:** 31 414,7 (Csoport 1) és 31 741,3 pont (Csoport 2), 1% különbség. Az összpontban a Csoport 2 előnye (476 120 a 376 976 ellen) főként a 3 fővel nagyobb létszámból adódik (5.3).
- **A munkamegosztás eltér:** a Csoport 2 egy főre kétszer annyi betárolási pontot szerzett (10 821,9 az 5 327,8 ellen), a Csoport 1 több szedésit (26 086,8 a 20 919,5 ellen).
- **Dolgozónként** a legtöbb és a legkevesebb pont aránya kb. 2:1 (Nagy Anna (8759) 44 609, Szabó Dóra (5627) 22 027; 5.2); a rangsor elején mindkét csoport tagjai szerepelnek.
- **Havonta** mindkét csoport májusban szerezte a legtöbb pontot (5.4). A június nem vethető össze közvetlenül a másik két hónappal, mert az adatok 06.23-ig tartanak (23 nap).
- **A 0 darabos sorok** pontjai a csoportpontok 1,7, illetve 1,3%-át adják; nélkülük sem változik a két csoport sorrendje (5.3, A-07).
- **A KPI nem a pontot használja:** a KPI betárolási és szedési része a kezelt darabszámon alapul (6. szakasz), ezért a két mutató eltérő képet adhat.

**Korlát:** munkaidő-adat nélkül a pontszám a könyvelt munka mennyiségét mutatja, nem a hatékonyságot; a dolgozók közötti különbségek okát (ledolgozott napok, beosztás) az adatból nem lehet megállapítani.

## 6. KPI

A feladat egyszerű KPI-rendszert kér: a 30 000 Ft-os kiinduló értéket a csoport létszáma alapján kell meghatározni, majd három azonos súlyú részre bontani, és a részeket a teljesítmény alapján arányosítani. A KPI-t csoportonként kell számolni, a Csoport 1 és a Csoport 2 tagjainak teljesítményét egymástól külön (D-08, D-09).

**A KPI felépítése (a feladat ábrája szerint):**

| Rész | Számítás | Döntés |
|---|---|---|
| Alap (létszám alapú) | 30 000 Ft × min(1, 12 / létszám): 12 fő fölött a létszám arányában csökken | D-10 |
| Komponens | az alap egyharmada; mindhárom rész legfeljebb ennyi lehet | – |
| Tárhely-rész | a teljes komponens | D-14 |
| Betárolás-rész | komponens × min(1, június / a 04–05–06 havi darabok átlaga) | D-11, D-12 a), D-13 |
| Szedés-rész | ugyanígy, a szedett darabokkal | D-11, D-12 a), D-13 |
| KPI | a három rész összege | – |

A min(1, x) a kettő közül a kisebbet adja: ha x nagyobb 1-nél, az eredmény 1.

- **Az alap a csoport KPI-jának felső határa:** a feladat nem mondja meg, hogy a 30 000 Ft egy főre vagy a csoportra vonatkozik; a számítás csoportszintű.
- **Tárhely-rész:** a tárhelyadat egyetlen pillanatkép (a legfrissebb állapot), havi alakulás nincs, ezért mindkét csoport a teljes egyharmadot kapja (D-14). A készlet bemutatása (feladat 2.a) a Power BI-ban, a 4.8 számaival.
- **Teljesítmény = kezelt darabszám** (feladat 3.a), ezért a betárolás- és a szedés-rész a darabot használja, nem a teljesítménypontot. A darabok a két csoport mozgássoraiból jönnek (`csoport_mozgas`, 5.1); a hónapot a lezárás napja adja (D-03).
- **A háromhavi átlag a 04, 05 és 06 hónapból áll, a júniusi összeg nyers** (06.23-ig, 23 nap), napi átlag nélkül (D-11, D-12 a). A feladat szövege „az azt megelőző három hónap” átlagát említi, de az adat csak áprilistól júniusig tart.
- **Ha a június eléri az átlagot, a rész teljes;** ha elmarad, a rész a június / átlag aránnyal csökken. Többletet nem ad: a szorzó legfeljebb 1 (D-13).
- **Érzékenység (6.4):** a) napi átlaggal, b) az 1 000 darabnál nagyobb sorok nélkül (D-12, D-19). A hivatalos KPI a 6.3-é.

**A szakasz lépései:** 6.1 alap és komponens; 6.2 havi darabok, átlag, arány, szorzó; 6.3 KPI csoportonként; 6.4 érzékenység; 6.5 az eredmény értelmezése.

### 6.1 Alap és komponens csoportonként (D-10)

A csoport létszáma az 5.3-ból jön (`LETSZAM`: 12 és 15 fő). A létszámszorzó 12 / létszám, felülről 1-nél korlátozva: 12 főnél 1, 15 főnél 0,8. Az alap 30 000 Ft × a létszámszorzó, a komponens az alap harmada. A cella új táblát indít (`kpi`, egy sor = egy csoport); a 6.3 ezt egészíti ki a három résszel és a KPI-val. A kontroll: a Csoport 1 alapja 30 000 Ft, a Csoport 2-é 24 000 Ft (30 000 × 12 / 15; D-10, a feladatkiíró 3. válasza).

In [30]:
# 6.1 Alap és komponens csoportonként: létszám alapú csökkentés, három egyenlő rész (D-10)
KPI_KIINDULO = 30_000  # a KPI kiinduló értéke, Ft (feladat 1.a)
MAX_LETSZAM = 12       # az engedélyezett maximális csoportlétszám (feladat 1.b)
KONTROLL_61 = {"Alap, Csoport 1": 30_000, "Alap, Csoport 2": 24_000}  # D-10: 30 000 × 12 / 15 = 24 000 (a feladatkiíró 3. válasza)

kpi = pd.DataFrame({"Letszam": LETSZAM}) # 2 sor; a sorcímke a csoport neve (5.3); szótárból tábla, a kulcs("Letszam") az oszlopnév, az érték egy sorozat (Series); a LETSZAM az 5.3-ban készült, és két elemből áll: „Csoport 1” → 12, „Csoport 2” → 15; a sorcímkét csak a 6.3 végén teszem oszlopba (reset_index()), ahogy az 5.3-ban a pont_csoport-nál. Addig a csoport neve alapján lehet illeszteni
kpi["Letszam_szorzo"] = (MAX_LETSZAM / kpi["Letszam"]).clip(upper=1)  # 12 / létszám, a .clip(upper=1): felső korlátot szab: minden 1 fölötti értéket 1-re cserél, a többihez nem nyúl (esetleg egy kitalált 10 fős csoport esetén 12/10=1.2 lenne) (feladat 1.c)
kpi["Alap"] = KPI_KIINDULO * kpi["Letszam_szorzo"] # új oszlop, a csoport KPI-jának felső határa
kpi["Komponens"] = kpi["Alap"] / 3   # komponens: az előbb számolt alap / 3 (feladat 1.d)

mert = {"Alap, Csoport 1": kpi.loc["Csoport 1", "Alap"], "Alap, Csoport 2": kpi.loc["Csoport 2", "Alap"]} # alapok kiírása
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_61[nev] else "ELTÉRÉS!"
    print(f"{nev:<16} {ertek:>7,.0f} (várt: {KONTROLL_61[nev]:>7,})  {allapot}")
kpi

Alap, Csoport 1   30,000 (várt:  30,000)  OK
Alap, Csoport 2   24,000 (várt:  24,000)  OK


,Letszam,Letszam_szorzo,Alap,Komponens
Csoport,,,,
Csoport 1,12,1.0,30000.0,10000.0
Csoport 2,15,0.8,24000.0,8000.0


### 6.2 Havi darabok, háromhavi átlag, arány és szorzó (D-11, D-12 a), D-13)

A két csoport mozgássorainak darabjai csoportonként, folyamatonként és hónaponként összegződnek: egy sor egy csoport egy folyamata (2 × 2 = 4 sor), az oszlopok a három hónap darabjai. Az átlag a három havi összeg átlaga, a júniussal együtt (D-11); a júniusi összeg nyers, a 06.23-ig tartó adatból (D-12 a). Az arány a júniusi összeg osztva az átlaggal; a szorzó az arány, felülről 1-nél korlátozva (D-13). A kontrollok: a raktár havi darabszáma (mind a négy kategória, 5.1) egyezik a munkanapló kontrollszámaival (6. fejezet); a négy sor darabjainak összege egyezik a Létszám tábla kódjaihoz tartozó mozgássorok darabjával, amelyet a cella csoportbesorolás és hónap nélkül, közvetlenül a FactMozgas-ból számol; a táblának 4 sora van.

In [31]:
# 6.2 Havi darabok csoportonként és folyamatonként; a június aránya a háromhavi átlaghoz és a szorzó (D-11, D-12 a, D-13)
HONAP_OSZLOPOK = ["Darab_04", "Darab_05", "Darab_06"] # a három havi darab oszlop nevének listája
KONTROLL_62 = {"Betárolás, április": 1_114_944, "Betárolás, május": 1_144_928, "Betárolás, június": 915_694,  # munkanapló, 6. fejezet: a raktár havi darabjai tisztítás után
               "Szedés, április":    1_129_983, "Szedés, május":    1_184_196, "Szedés, június":    896_005,
               "A két csoport darabja": FactMozgas.loc[FactMozgas["ugyintezo"].isin(letszam["ugyintezo_kod"]), "darab"].sum(),  # független út (mint az 5.1-ben): a Létszám tábla 27 kódjának darabjai, csoport és hónap nélkül
               "Sorok (2 csoport × 2 folyamat)": 4}

osszes_havi = mozgas.pivot_table(index="statusz", columns="Honap", values="darab", aggfunc="sum")  # a raktár havi darabjai, mind a négy kategória (5.1); csak a kontrollhoz

kpi_havi = csoport_mozgas.pivot_table(index=["Csoport", "statusz"], columns="Honap", values="darab", aggfunc="sum").reset_index()  # sor: csoport × folyamat, oszlop: hónap, cella: a darabok összege; .reset_index() miatt a csoport és a statusz sorcímkéből oszlop lesz
kpi_havi = kpi_havi.rename(columns={4: "Darab_04", 5: "Darab_05", 6: "Darab_06"})  # szöveges oszlopnevek (a Parquet nem fogad számot oszlopnévként), a szótár kulcsai itt számok, nem idézőjeles szövegek, mert a pivot oszlopnevei számok (4,5,6) -> a 4: "Darab_04" jelentése: a 4 nevű oszlop neve „Darab_04” legyen (a Parquet csak szöveges oszlopnevet fogad el)
kpi_havi.columns.name = None                                    # a pivotból maradt tengelynév törlése („Honap”)
kpi_havi["Atlag"]  = kpi_havi[HONAP_OSZLOPOK].mean(axis=1)      # axis=1: soronként(vízszintesen) a három havi összeg átlaga (D-11)
kpi_havi["Arany"]  = kpi_havi["Darab_06"] / kpi_havi["Atlag"]   # a nyers júniusi összeg az átlaghoz képest (nyers június 23 nappal, ezért eleve kisebb) (D-12 a)
kpi_havi["Szorzo"] = kpi_havi["Arany"].clip(upper=1)            # legfeljebb 1: többletet nem ad (D-13)

mert = {"Betárolás, április": osszes_havi.loc["BETAROLAS", 4], "Betárolás, május": osszes_havi.loc["BETAROLAS", 5], "Betárolás, június": osszes_havi.loc["BETAROLAS", 6],
        "Szedés, április":    osszes_havi.loc["SZEDES", 4],    "Szedés, május":    osszes_havi.loc["SZEDES", 5],    "Szedés, június":    osszes_havi.loc["SZEDES", 6],
        "A két csoport darabja": kpi_havi[HONAP_OSZLOPOK].sum().sum(), # első sum oszloponként ad három számot, a második ezeket adja össze
        "Sorok (2 csoport × 2 folyamat)": len(kpi_havi)}  #sorok száma, 4
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_62[nev] else "ELTÉRÉS!"
    print(f"{nev:<31} {ertek:>9,} (várt: {KONTROLL_62[nev]:>9,})  {allapot}")
kpi_havi.round(4)

Betárolás, április              1,114,944 (várt: 1,114,944)  OK
Betárolás, május                1,144,928 (várt: 1,144,928)  OK
Betárolás, június                 915,694 (várt:   915,694)  OK
Szedés, április                 1,129,983 (várt: 1,129,983)  OK
Szedés, május                   1,184,196 (várt: 1,184,196)  OK
Szedés, június                    896,005 (várt:   896,005)  OK
A két csoport darabja           1,118,240 (várt: 1,118,240)  OK
Sorok (2 csoport × 2 folyamat)          4 (várt:         4)  OK


,Csoport,statusz,Darab_04,Darab_05,Darab_06,Atlag,Arany,Szorzo
0,Csoport 1,BETAROLAS,43738,49414,47489,46880.3333,1.013,1.0
1,Csoport 1,SZEDES,104194,107873,94855,102307.3333,0.9272,0.9272
2,Csoport 2,BETAROLAS,108871,130352,112858,117360.3333,0.9616,0.9616
3,Csoport 2,SZEDES,111618,114987,91991,106198.6667,0.8662,0.8662


### 6.3 KPI csoportonként (D-13, D-14)

A betárolás- és a szedés-rész a komponens (6.1) és a szorzó (6.2) szorzata, forintban; ez a `kpi_havi` táblába kerül. A tárhely-rész mindkét csoportnak a teljes komponens (D-14). A három rész a `kpi` táblába kerül (6.1), a KPI ezek összege, az elmaradás az alap és a KPI különbsége. A táblákban pontos forintértékek vannak, kerekítés csak a kiírásnál. A kontrollok: a KPI egyezik a tervezéskor külön szkripttel mért értékkel (29 271,57 és 22 622,82 Ft); a Csoport 2 szedés-részét a cella egy második úton is kiszámolja, közvetlenül a mozgássorokból (szűrés és hónaponkénti összeg, pivot nélkül); a KPI egyik csoportnál sem nagyobb az alapnál.

In [32]:
# 6.3 KPI csoportonként: a három rész forintban, a KPI és az elmaradás (D-13, D-14)
cs2_szedes = csoport_mozgas.loc[(csoport_mozgas["Csoport"] == "Csoport 2") & (csoport_mozgas["statusz"] == "SZEDES")]  # második út: a Csoport 2 szedési sorai
cs2_honap = cs2_szedes.groupby("Honap")["darab"].sum()   # 3 sor: a három hónap darabjai 4->111 618, 5->..., 6->...
cs2_szedes_resz = kpi.loc["Csoport 2", "Komponens"] * min(1, cs2_honap[6] / cs2_honap.mean())  # 8 000 × min(1, június / átlag) -> 8000 × (min(1, a 6-os sorcímkéjű elem) / a hónapok átlaga); a sorcímkék 4,5,6, azaz a hónapok számai
KONTROLL_63 = {"KPI, Csoport 1": 29_271.57, "KPI, Csoport 2": 22_622.82,  # a 6. szakasz tervének mért számai (10-04)
               "Szedés-rész, Csoport 2": round(cs2_szedes_resz, 2)}  # 2 tizedesjegyes kerekítés

kpi_havi["Komponens"] = kpi_havi["Csoport"].map(kpi["Komponens"])  # a csoport komponense (6.1): Csoport 1 → 10 000, Csoport 2 → 8 000
kpi_havi["Resz"] = kpi_havi["Komponens"] * kpi_havi["Szorzo"]       # a betárolás- vagy a szedés-rész, Ft

reszek = kpi_havi.pivot_table(index="Csoport", columns="statusz", values="Resz", aggfunc="sum")  # 2 × 2: sor a csoport, oszlop a folyamat, cella a rész Ft-ja, átmeneti tábla
kpi["Keszlet_resz"]   = kpi["Komponens"]     # a tárhely-rész: a teljes komponens (D-14)
kpi["Betarolas_resz"] = reszek["BETAROLAS"]  # sorcímke szerinti illesztés (5.3)
kpi["Szedes_resz"]    = reszek["SZEDES"]
kpi["KPI"]       = kpi["Keszlet_resz"] + kpi["Betarolas_resz"] + kpi["Szedes_resz"] # a három rész összege
kpi["Elmaradas"] = kpi["Alap"] - kpi["KPI"]

mert = {"KPI, Csoport 1": round(kpi.loc["Csoport 1", "KPI"], 2), "KPI, Csoport 2": round(kpi.loc["Csoport 2", "KPI"], 2),
        "Szedés-rész, Csoport 2": round(kpi.loc["Csoport 2", "Szedes_resz"], 2)}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_63[nev] else "ELTÉRÉS!"
    print(f"{nev:<23} {ertek:>9,.2f} (várt: {KONTROLL_63[nev]:>9,.2f})  {allapot}")
alap_ok = (kpi["KPI"] <= kpi["Alap"]).all()
print(f"A KPI egyik csoportnál sem nagyobb az alapnál: {'OK' if alap_ok else 'ELTÉRÉS!'}")

kpi = kpi.reset_index()  # a Csoport a sorcímkéből oszlop lesz (az exporthoz)
kpi.round(2)

KPI, Csoport 1          29,271.57 (várt: 29,271.57)  OK
KPI, Csoport 2          22,622.82 (várt: 22,622.82)  OK
Szedés-rész, Csoport 2   6,929.73 (várt:  6,929.73)  OK
A KPI egyik csoportnál sem nagyobb az alapnál: OK


,Csoport,Letszam,Letszam_szorzo,Alap,Komponens,Keszlet_resz,Betarolas_resz,Szedes_resz,KPI,Elmaradas
0,Csoport 1,12,1.0,30000.0,10000.0,10000.0,10000.0,9271.57,29271.57,728.43
1,Csoport 2,15,0.8,24000.0,8000.0,8000.0,7693.09,6929.73,22622.82,1377.18


### 6.4 Érzékenység: napi átlag és kiugrók (D-12, D-19)

A hivatalos KPI a 6.3-é. Ez a cella két másik számítási módot mutat, hogy látszódjon, mennyire függ az eredmény a módszertani döntésektől:

- **a) Napi átlag (a D-12 eredeti javaslata):** a havi darab osztva a raktár adott havi működési napjainak számával, folyamatonként (azok a napok, amelyeken a raktárban bárki lezárt betárolási, illetve szedési sort). Így a 23 napos június összevethető a 29–30 napos áprilissal és májussal. A feladatkiíró kérése szerint a hivatalos számítás nem így készül (D-12 a).
- **b) Kiugrók nélkül (D-19):** a 6.2 számítása az 1 000 darabnál nagyobb sorok nélkül; a két csoportban 5 ilyen sor van, mind a Csoport 2 szedési sora áprilisban és májusban.

A kontrollok: a működési napok száma egyezik a munkanapló kontrollszámaival (betárolás 29 / 29 / 23, szedés 30 / 30 / 23; 6. fejezet); a kiugró sorok száma 5 (D-19); a kiugrók nélküli és a kiugró sorok darabjának összege egyezik a 6.2 darabszámával.

In [33]:
# 6.4 Érzékenység: a) napi átlaggal (D-12), b) az 1 000 darab feletti sorok nélkül (D-19); a hivatalos KPI a 6.3-é
KIUGRO = 1_000                                 # D-19: az ennél több darabos sor kiugró
kiugro_sor = csoport_mozgas["darab"] > KIUGRO  # True a kiugró sorokon (mind a két csoportban)
KONTROLL_64 = {"Napok, betárolás, április": 29, "Napok, betárolás, május": 29, "Napok, betárolás, június": 23,  # munkanapló, 6. fejezet
               "Napok, szedés, április":    30, "Napok, szedés, május":    30, "Napok, szedés, június":    23,
               "Kiugró sorok a két csoportban": 5,               # D-19
               "Darab: kiugrók nélkül + kiugrók": kpi_havi[HONAP_OSZLOPOK].sum().sum()}   # = a 6.2 darabja

# a) napi átlag: a havi darab osztva a raktár működési napjaival
napok = mozgas.pivot_table(index="statusz", columns="Honap", values="Datum", aggfunc="nunique")  # sor: folyamat, oszlop: hónap, cella: hány különböző napon volt lezárt sor a raktárban; aggfunc="nunique": a nunique itt összesítő függvényként áll(aggfunc miatt), és cellánként a különböző dátumok számát adja; a tábla a teljes mozgas-ból készül, mind a 2,96 millió sorból, vagyis a raktár egészéből
erz_havi = kpi_havi[["Csoport", "statusz", "Komponens", "Arany"]] # új tábla a kpi_havi felsorolt oszlopaiból, így a kpi_havi nem módosul, azaz hivatalos marad számunkra
erz_havi["Napi_04"] = kpi_havi["Darab_04"] / kpi_havi["statusz"].map(napok[4])  # áprilisi darab / a folyamat áprilisi működési napjai; .map(...): minden kpi_havi sor a saját folyamatának áprilisi napszámát kapja
erz_havi["Napi_05"] = kpi_havi["Darab_05"] / kpi_havi["statusz"].map(napok[5])  # májusi darab / a folyamat májusi működési napjai
erz_havi["Napi_06"] = kpi_havi["Darab_06"] / kpi_havi["statusz"].map(napok[6])  # júni
erz_havi["Arany_a"] = erz_havi["Napi_06"] / erz_havi[["Napi_04", "Napi_05", "Napi_06"]].mean(axis=1) # június napi értékek átlaga / három havi napi átlag = arány

# b) kiugrók nélkül: a 6.2 számítása a legfeljebb 1 000 darabos sorokból
havi_b = csoport_mozgas.loc[csoport_mozgas["darab"] <= KIUGRO].pivot_table(index=["Csoport", "statusz"], columns="Honap", values="darab", aggfunc="sum").reset_index() # a 6.2 pivotja plusz egy szűréssel előtte, csak a max 1000 darabos sorokat tartja meg
havi_b["Arany_b"] = havi_b[6] / havi_b[[4, 5, 6]].mean(axis=1)
erz_havi = erz_havi.merge(havi_b[["Csoport", "statusz", "Arany_b"]], how="left", on=["Csoport", "statusz"], validate="one_to_one")

erz_havi["Resz_a"] = erz_havi["Komponens"] * erz_havi["Arany_a"].clip(upper=1) #komponens × arány (max: 1)
erz_havi["Resz_b"] = erz_havi["Komponens"] * erz_havi["Arany_b"].clip(upper=1)
resz_a = erz_havi.groupby("Csoport")["Resz_a"].sum()  # csoportonként a betárolás- és a szedés-rész összege, a) változat
resz_b = erz_havi.groupby("Csoport")["Resz_b"].sum()  # csoportonként a betárolás- és a szedés-rész összege, b) változat

erzekenyseg = kpi[["Csoport", "Alap", "KPI"]]
erzekenyseg["KPI_a_napi_atlag"]     = kpi["Keszlet_resz"] + kpi["Csoport"].map(resz_a)
erzekenyseg["KPI_b_kiugrok_nelkul"] = kpi["Keszlet_resz"] + kpi["Csoport"].map(resz_b)

mert = {"Napok, betárolás, április": napok.loc["BETAROLAS", 4], "Napok, betárolás, május": napok.loc["BETAROLAS", 5], "Napok, betárolás, június": napok.loc["BETAROLAS", 6],
        "Napok, szedés, április":    napok.loc["SZEDES", 4],    "Napok, szedés, május":    napok.loc["SZEDES", 5],    "Napok, szedés, június":    napok.loc["SZEDES", 6],
        "Kiugró sorok a két csoportban": kiugro_sor.sum(),
        "Darab: kiugrók nélkül + kiugrók": havi_b[[4, 5, 6]].sum().sum() + csoport_mozgas.loc[kiugro_sor, "darab"].sum()}
for nev, ertek in mert.items():
    allapot = "OK" if ertek == KONTROLL_64[nev] else "ELTÉRÉS!"
    print(f"{nev:<32} {ertek:>9,} (várt: {KONTROLL_64[nev]:>9,})  {allapot}")
print(f"\nA kiugró sorok darabja: {csoport_mozgas.loc[kiugro_sor, 'darab'].sum():,}")
print("\nArányok (június / átlag): hivatalos, a) napi átlaggal, b) kiugrók nélkül:")
print(erz_havi[["Csoport", "statusz", "Napi_04", "Napi_05", "Napi_06", "Arany", "Arany_a", "Arany_b"]].round(4))
erzekenyseg.round(2)

Napok, betárolás, április               29 (várt:        29)  OK
Napok, betárolás, május                 29 (várt:        29)  OK
Napok, betárolás, június                23 (várt:        23)  OK
Napok, szedés, április                  30 (várt:        30)  OK
Napok, szedés, május                    30 (várt:        30)  OK
Napok, szedés, június                   23 (várt:        23)  OK
Kiugró sorok a két csoportban            5 (várt:         5)  OK
Darab: kiugrók nélkül + kiugrók  1,118,240 (várt: 1,118,240)  OK

A kiugró sorok darabja: 9,944

Arányok (június / átlag): hivatalos, a) napi átlaggal, b) kiugrók nélkül:
     Csoport    statusz    Napi_04    Napi_05    Napi_06   Arany  Arany_a  \
0  Csoport 1  BETAROLAS  1508.2069   1703.931  2064.7391   1.013   1.1738   
1  Csoport 1     SZEDES  3473.1333  3595.7667  4124.1304  0.9272   1.1054   
2  Csoport 2  BETAROLAS  3754.1724  4494.8966  4906.8696  0.9616   1.1189   
3  Csoport 2     SZEDES     3720.6     3832.9  3999.6087  0.8662  

,Csoport,Alap,KPI,KPI_a_napi_atlag,KPI_b_kiugrok_nelkul
0,Csoport 1,30000.0,29271.57,30000.0,29271.57
1,Csoport 2,24000.0,22622.82,24000.0,22846.08


### 6.5 A KPI értelmezése

- **Eredmény:** a Csoport 1 KPI-ja 29 271,57 Ft a 30 000 Ft-os alapból (97,6%), a Csoport 2-é 22 622,82 Ft a 24 000 Ft-ból (94,3%). Az elmaradás 728,43, illetve 1 377,18 Ft (6.3).
- **Az alapok különbsége a létszámból jön:** a Csoport 2 15 fős, ezért az alapja 30 000 × 12 / 15 = 24 000 Ft (6.1). A két KPI forintban ezért közvetlenül nem vethető össze, az alaphoz mért arányuk (97,6 és 94,3%) igen.
- **A tárhely-rész** mindkét csoportnál teljes (10 000 és 8 000 Ft), mert a készlet pillanatkép, nincs mihez mérni (D-14).
- **Hol marad el:** a Csoport 1 betárolása júniusban az átlag fölött volt (1,013, teljes rész), csak a szedése maradt el (0,927; −728,43 Ft). A Csoport 2 mindkét folyamatban elmaradt: a betárolás 0,962 (−306,91 Ft), a szedés 0,866 (−1 070,27 Ft) (6.2, 6.3).
- **Az elmaradás oka a rövid június:** a júniusi adat 06.23-ig tart (23 működési nap a 29–30 helyett), ezért a nyers havi összeg eleve kisebb. Napi átlagban mind a négy arány 1 fölött van (1,039–1,174): napi tempóban mindkét csoport mindkét folyamatban júniusban dolgozott a leggyorsabban, és ezzel a módszerrel mindkét csoport a teljes alapot kapná (6.4 a). A hivatalos számítás a feladatkiíró kérésére a nyers összeggel készül (D-12 a).
- **A kiugró sorok hatása kicsi:** az 5 darab, 1 000 darabnál nagyobb sor (mind a Csoport 2 áprilisi és májusi szedése) az összehasonlítási alapot növeli; nélkülük a Csoport 2 szedési aránya 0,866-ról 0,894-re nő, a KPI-ja 223,26 Ft-tal, 22 846,08 Ft-ra (6.4 b, D-19).
- **A KPI és a teljesítménypont mást mér:** egy főre jutó pontban a Csoport 2 kicsit előrébb van (31 741,3 a 31 414,7 ellen; 5.3), a KPI alaphoz mért arányában a Csoport 1 (97,6 a 94,3% ellen). A pont a három hónap könyvelt munkáját méri tételsorokban, a KPI a júniusi darabszámot a háromhavi átlaghoz.

**Korlát:** a KPI a részleges júniust teljes hónapokhoz méri, ezért a rövidebb hónap akkor is csökkenti a KPI-t, ha a napi teljesítmény nem romlott. A kezelt darabszám a beérkező áru és a rendelések mennyiségétől is függ, nem csak a csoport munkájától. Javaslat a jövőre: teljes hónapok összevetése, vagy napi átlag (6.4 a).

## 7. Export a Power BI-nak

A modell öt táblája és az 5–6. szakasz öt eredménytáblája (PontDolgozo, PontCsoport, PontHavi, KPI, KPIHavi) Parquet formátumban kerül a `06_Kimenet` mappába, táblánként egy fájlba (D-26). A Power BI a kész eredménytáblákat tölti be, és a pontokat és a KPI-t DAX-ban is kiszámolja, kontrollként (D-24 a). A Parquet a típusokat is tárolja (szöveg, egész szám, tizedes tört, dátum/idő, igaz/hamis), így a Power BI a vezető nullás kódokat szövegként, az időpontokat dátum/időként olvassa be, a gép területi beállításától függetlenül (D-21). Az eredménytáblákban a csoport és a dolgozó oszlopként szerepel (`reset_index()`, 5.2–6.3), és minden oszlopnév szöveg, mert a Parquet számot nem fogad el oszlopnévként. A cella minden fájlt visszaolvas, és összeveti a notebookbeli táblával: ha az értékek és a típusok is egyeznek, az export hiánytalan. Az érzékenység (6.4) nem kerül az exportba.

In [34]:
# 7. Export: a modell táblái és az 5–6. szakasz eredménytáblái Parquet-fájlokba, táblánként egy fájl (D-26); visszaolvasással ellenőrizve
KIMENET = MAPPA / "06_Kimenet"
KIMENET.mkdir(exist_ok=True)  # létrehozza a mappát, ha még nincs

export_tablak = {"FactMozgas": FactMozgas, "FactKeszlet": FactKeszlet,
                 "DimDolgozo": DimDolgozo, "DimCikk": DimCikk, "DimDatum": DimDatum,
                 "PontDolgozo": pont_dolgozo, "PontCsoport": pont_csoport, "PontHavi": pont_havi,  # az 5. szakasz eredménytáblái (5.2–5.4)
                 "KPI": kpi, "KPIHavi": kpi_havi}                                                  # a 6. szakasz eredménytáblái (6.1–6.3)

for nev, tabla in export_tablak.items():
    fajl = KIMENET / f"{nev}.parquet"  # a fájl bekerül a KIMENET adta mappába, a táblanévvel és a .parquet kiterjesztéssel
    tabla.to_parquet(fajl, index=False)  # a táblát a Parquet-fájlba írja, index=False: a sorcímkék nem kerülnek a fájlba
    vissza = pd.read_parquet(fajl)  # a fájlt visszaolvassa egy új táblába, hogy ellenőrizze az írás-olvasás folyamatát
    allapot = "OK" if vissza.equals(tabla) else "ELTÉRÉS!"  # True, ha a két tábla teljesen azonos, beleértve az oszlopok sorrendjét is
    print(f"{nev:<12} {len(vissza):>10,} sor  {len(vissza.columns):>2} oszlop  {fajl.stat().st_size / 1_000_000:>6.1f} MB  {allapot}")

FactMozgas    2,957,772 sor   9 oszlop    58.4 MB  OK
FactKeszlet     834,925 sor  10 oszlop     8.6 MB  OK
DimDolgozo          559 sor   5 oszlop     0.0 MB  OK
DimCikk         450,149 sor   9 oszlop     6.0 MB  OK
DimDatum             91 sor   9 oszlop     0.0 MB  OK
PontDolgozo          27 sor   6 oszlop     0.0 MB  OK
PontCsoport           2 sor   9 oszlop     0.0 MB  OK
PontHavi              6 sor   9 oszlop     0.0 MB  OK
KPI                   2 sor  10 oszlop     0.0 MB  OK
KPIHavi               4 sor  10 oszlop     0.0 MB  OK
